# Causal Fairness in Synthetic Data Generation — findings so far

**What this project asks.** When you replace real data with *synthetic* data, you get to
choose the generating process. That is an unusual amount of leverage: you can build the
data so that the unfair pathways simply are not there. This project asks whether that
leverage is real — can you remove a causal pathway from protected attribute to outcome
*at generation time*, and does a model trained on the resulting data actually come out
fairer, without wrecking the data's usefulness or its privacy guarantee?

**What is in this notebook.** Every number here comes from 2,040 completed experiment runs
(23.5 recorded machine-hours; the GAN re-run cost a further ~18 h of wall-clock across two
GPUs, most of it absorbed by the training cache). A further 960 superseded runs ship in the
same file, used only in
Section 4.3 to show what the correction changed. Sections are collapsible — click the
triangle next to a heading to fold it away.

| Section | What you get |
|---|---|
| 1. The experiment | The grid: datasets, methods, knobs |
| 2. Metrics, in plain language | **What each number actually means.** Read this once |
| 3. Who counts as protected | The sensitive / admissible attribute splits |
| 4. Data quality | Does the synthetic data look like the real data? |
| 5. Privacy | What the privacy budget costs you |
| 6. Usefulness | Can you train a model on it? |
| 7. Fairness | The main event — do the mechanisms work? |
| 8. GAN backbones | A new result on making causal GANs private |
| 9. Takeaways | What we can claim today |
| 10. Next steps | What to run next, and the paper case |

---

### The five-line version

1. **Fairness mechanisms work, and they are nearly free.** Blocking the protected→outcome
   pathway cuts the downstream demographic-parity gap substantially, and costs almost
   nothing in accuracy or fidelity. That is the core claim of the project and it holds up.
2. **The mechanism matters more than the generator.** Which SDG method you use changes
   fidelity a lot; which fairness mechanism you use changes fairness a lot. They are
   surprisingly separable, which is good news for a general framework.
3. **Privacy and fairness are not the trade-off people assume.** Across marginal-based
   methods, tightening ε from 1000 to 1 barely moves the fairness gap. The two knobs are
   close to independent.
4. **Differentially private causal GANs mostly collapse — unless you fix the
   representation.** Standard DP-GAN training of DECAF's causal generator produces
   single-class garbage in 47% of runs. Swapping in a CTGAN-style representation fixes it
   completely: **0 collapses out of 360 runs**, and 0 out of 720 counting both epoch
   settings ever run. This is new and is the strongest
   *methods* contribution here.
5. **Accuracy needs a reference line.** Several methods score "75% accuracy" on Adult
   while a model that always guesses the majority class scores 74.7%. Section 6 reports
   lift over that trivial baseline instead, and it changes the ranking.

## Setup

Run this once. It loads the results table and defines the plotting helpers.

In [ ]:
import warnings, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 200)

# ---- find the results, unzipping them if that is all that is present ----
# Colab workflow: drag CausalFairnessInSDG_results.zip into the file pane and
# run this cell. Nothing else to do -- it finds the zip, unpacks it, and goes.
import zipfile, glob

SEARCH = [Path('.'), Path('data'), Path('report/data'), Path('/content'),
          Path('/content/data'), Path('results')]

def _find_data():
    for p in SEARCH:
        if (p / 'all_runs.csv').exists():
            return p
    return None

DATA = _find_data()
if DATA is None:
    zips = sorted({z for d in ['.', '/content'] for z in glob.glob(d + '/*.zip')})
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            if not any(n.endswith('all_runs.csv') for n in zf.namelist()):
                continue
            print('unzipping %s ...' % z)
            zf.extractall('.')
    DATA = _find_data()
    # The zip may carry its own directory prefix (results/, report/data/...),
    # so fall back to locating the CSV wherever it landed.
    if DATA is None:
        hits = glob.glob('**/all_runs.csv', recursive=True)
        DATA = Path(hits[0]).parent if hits else None
if DATA is None:
    try:  # last resort in Colab: ask for the zip directly
        from google.colab import files
        print('Upload CausalFairnessInSDG_results.zip')
        up = files.upload()
        for name in up:
            if name.endswith('.zip'):
                zipfile.ZipFile(name).extractall('.')
        hits = glob.glob('**/all_runs.csv', recursive=True)
        DATA = Path(hits[0]).parent if hits else None
    except ImportError:
        pass
if DATA is None:
    raise FileNotFoundError(
        'all_runs.csv not found. Put CausalFairnessInSDG_results.zip next to '
        'this notebook (or unzip it there) and re-run this cell.')
print('results loaded from: %s' % DATA.resolve())

runs_all = pd.read_csv(DATA / 'all_runs.csv')
baselines = pd.read_csv(DATA / 'real_baselines.csv')

# ---- reference lines from the REAL data (per dataset, averaged over 5 splits) ----
REF = baselines.groupby('dataset').mean(numeric_only=True)
MAJORITY = REF['majority_baseline'].to_dict()   # accuracy of 'always guess most common'
TRTR = REF['trtr_mlp'].to_dict()                # accuracy of training on REAL data

# ---- derived columns (on the full file, so both views get them) ----
runs_all['acc'] = runs_all['downstream_accuracy_mlp']
runs_all['collapsed'] = runs_all['status'] == 'partial'
runs_all['lift'] = runs_all['acc'] - runs_all['dataset'].map(MAJORITY)      # over trivial predictor
runs_all['acc_retained'] = runs_all['acc'] / runs_all['dataset'].map(TRTR)  # vs training on real
runs_all['eps_label'] = runs_all['epsilon'].map(
    lambda e: 'none' if pd.isna(e) else ('%g' % e))

# ---- supersession ----
# The GAN family was re-run at corrected epoch settings (batch
# gan-retuned-2026-08-04) after the original epochs were found to have been
# tuned on a metric that cannot rank configurations -- see Section 4.3. The
# superseded rows are kept in the CSV so Section 4.3 can show the before/after
# directly, but every other section analyses the ACTIVE rows only.
runs = runs_all[~runs_all['superseded']].reset_index(drop=True)
superseded = runs_all[runs_all['superseded']].reset_index(drop=True)

# ---- presentation ----
METHOD_ORDER = ['mst', 'privbayes', 'privsyn', 'decaf',
                'decaf_ctgan', 'decaf_dpgan', 'decaf_dpctgan']
MECH_ORDER = ['none', 'ftu', 'dp', 'cf']
EPS_ORDER = ['1', '10', '1000', 'none']
DATASETS = ['adult', 'compas']
NICE = {'mst': 'MST', 'privbayes': 'PrivBayes', 'privsyn': 'PrivSyn', 'decaf': 'DECAF',
        'decaf_ctgan': 'DECAF+CTGAN', 'decaf_dpgan': 'DECAF+DP-GAN',
        'decaf_dpctgan': 'DECAF+DP-CTGAN', 'adult': 'Adult', 'compas': 'COMPAS',
        'snake': 'SNAKE', 'sbo': 'SBO',
        'none': 'none', 'ftu': 'FTU', 'dp': 'DP', 'cf': 'CF'}
COLORS = {'mst': '#4C72B0', 'privbayes': '#DD8452', 'privsyn': '#55A868',
          'decaf': '#C44E52', 'decaf_ctgan': '#8172B3', 'decaf_dpgan': '#937860',
          'decaf_dpctgan': '#DA8BC3'}
MECH_COLORS = {'none': '#BBBBBB', 'ftu': '#4C72B0', 'dp': '#C44E52', 'cf': '#55A868'}

mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 13,
                     'axes.titleweight': 'bold', 'axes.grid': True,
                     'grid.alpha': 0.25, 'axes.axisbelow': True,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': 'white'})

def order(df, col, seq):
    """Sort a frame by a known category order, keeping only present values."""
    seq = [v for v in seq if v in set(df[col])]
    return df.set_index(col).reindex(seq).reset_index()

def nice(s):
    return NICE.get(s, s)

def bar_labels(ax, fmt='%.3f', pad=3, size=9):
    for c in ax.containers:
        ax.bar_label(c, fmt=fmt, padding=pad, fontsize=size)

# ---- sanity check ----
print('rows in file    :', len(runs_all))
print('active runs     :', len(runs), ' (superseded, kept for S4.3: %d)' % len(superseded))
print('batches (active):', runs.batch.value_counts().to_dict())
print('status  (active):', runs.status.value_counts().to_dict())
print()
print('Reference accuracy on the REAL data (what synthetic data is competing with):')
display(REF[['majority_baseline', 'trtr_mlp', 'trtr_lr', 'trtr_rf',
             'n_rows', 'n_train', 'n_holdout', 'base_rate_pos']].round(4))

## 1. The experiment

Every run is one cell of a grid. One cell = *pick a dataset, pick a generator, pick a
privacy budget, pick a fairness mechanism, pick who counts as protected, pick a random
seed* → generate a synthetic table → measure everything.

| Axis | Values | n |
|---|---|---|
| Dataset | Adult (30,162 rows, 14 attrs), COMPAS (6,172 rows, 8 attrs) | 2 |
| SDG method | MST, PrivBayes, PrivSyn, DECAF, DECAF+CTGAN, DECAF+DP-GAN, DECAF+DP-CTGAN | 7 |
| Privacy budget ε | 1, 10, 1000 (non-private methods: n/a) | 3 |
| Fairness mechanism | none, FTU, DP, CF | 4 |
| Protected/admissible split | 3 per dataset (Section 3) | 3 |
| Seed | 0–4 | 5 |

**2,040 active runs.** Everything is repeated 5 times with different seeds, because the
seed-to-seed noise turned out to be large enough that single runs are not trustworthy —
quantified in Section 7.4.

### The two families of generator

**Marginal-based (MST, PrivBayes, PrivSyn, AIM).** These pick a set of low-order marginals
(counts of one or two columns at a time), measure them under differential privacy, and then
fit a graphical model that reproduces those noisy counts. They are the state of the art for
private tabular data and they are *very* good at fidelity.

**Causal-GAN (DECAF and its variants).** DECAF trains a GAN whose generator is factored to
follow a causal DAG: each variable gets its own sub-network that can only see its parents.
That structure is what makes causal fairness possible — you can remove an edge and the
generator physically cannot use it.

> **Note on scope.** AIM is implemented and registered in the codebase but was left out of
> both batches. It should be in the next one.

### 1.1 How big is the causal graph? (and the two datasets being added)

Every fairness mechanism here works by **cutting or rerouting edges** in a causal DAG. So the
size of that DAG is not a detail — it is the thing being operated on. With only Adult and
COMPAS we cannot separate *"this mechanism works"* from *"the graph was small enough that
nearly every protected→outcome path was direct"*.

The last column is the one that matters: **how many distinct causal routes run from a
protected attribute to the outcome.** That is the number of things a mechanism has to find
and block, and it grows far faster than the attribute count.

| Dataset | Rows | Attributes | DAG edges | Protected→outcome paths | Status |
|---|---|---|---|---|---|
| **COMPAS** | 6,172 | **8** | 16 | **15** | in the results below |
| **Adult** | 30,162 | **14** | 26 | **37** | in the results below |
| **SNAKE** | 50,000 | **15** | 39 | **76** | ⏳ added, batch running |
| **SBO** (NIST) | 50,000 | **25** | 68 | **465** | ⏳ added, batch running |

**COMPAS has 8 attributes** — that was the direct question. It is the smallest table here and
its graph is correspondingly thin.

**SNAKE** (a Current Population Survey extract, 15 attributes) is a familiar income-prediction
task like Adult, so its fairness numbers stay interpretable, but with a longer education →
occupation → industry → hours → income chain and a parallel family-structure pathway. Its
outcome is family income over \$75k; protected attributes are sex, race and citizenship.

**SBO** (NIST Survey of Business Owners, 25 of its 133 attributes) is the interesting one, and
deliberately a *different kind* of fairness question: the protected attributes describe the
business **owner** (sex, race, ethnicity, veteran status, place of birth) while the outcome
describes the **business** (receipts above median). Every protected→outcome path therefore
runs through the firm — through sector, firm age, employment, web presence — which is exactly
the regime where CF (block only the paths *not* routed through admissible attributes) should
visibly separate from DP (block them all). On COMPAS the two are hard to tell apart because
almost every path is one or two edges long. With **465 paths**, they should not be.

> The remaining ~108 SBO columns are near-constant yes/no funding-source and language flags.
> They were dropped on purpose: they would widen the table without adding causal structure,
> and the point of including SBO is a **bigger graph**, not a wider one.

> **Status:** both datasets are implemented, their DAGs validate, and their first batch has
> **finished** — 216 runs, MST/PrivBayes/PrivSyn, 1 seed. Those results are in **Section
> 7.8**, which is where the scaling question this table sets up gets its first answer (short
> version: SBO gives the best data quality in the project, and the *worst* fairness-mechanism
> reliability). Sections 4–6 and 8 are still Adult and COMPAS only, and no DECAF variant has
> run on the new datasets yet.

In [ ]:
# The grid as actually run (rows = completed runs per cell)
g = (runs.groupby(['dataset', 'sdg_method', 'eps_label'])
          .agg(runs=('run_id', 'size'),
               collapsed=('collapsed', 'sum'),
               mechanisms=('fairness_mechanism', 'nunique'),
               role_splits=('role_config', 'nunique'),
               seeds=('seed', 'nunique'),
               minutes=('duration_seconds', lambda s: round(s.sum() / 60, 1)))
          .reset_index())
g['sdg_method'] = pd.Categorical(g.sdg_method, METHOD_ORDER, ordered=True)
g['eps_label'] = pd.Categorical(g.eps_label, EPS_ORDER, ordered=True)
g = g.sort_values(['dataset', 'sdg_method', 'eps_label'])
print('Total runs: %d   |   Total compute: %.1f hours'
      % (len(runs), runs.duration_seconds.sum() / 3600))
g

## 2. Metrics, in plain language

This is the reference section. Each metric gets: what it literally computes, what it is
*really* telling you about the data, and — importantly — what it is blind to.

> ### ⚠️ One naming collision to get straight first
> **"DP" means two completely different things in this project.**
> - **DP = Differential Privacy** (the ε budget). A *privacy* property of the generator.
> - **DP = Demographic Parity** (a fairness mechanism, and a fairness metric). A *fairness*
->   property of a downstream model.

> They are unrelated. Wherever it is ambiguous below, it is spelled out.

### 2.1 Fidelity — "does the fake data look like the real data?"

#### 1-way TVD  (`tvd_1way`) ↓ lower is better

For one column, take the real histogram $p$ and the synthetic histogram $q$ and compute
the **total variation distance**:

$$\mathrm{TVD}(p,q) = \tfrac{1}{2}\sum_{v}|p(v) - q(v)|$$

Then average over all columns.

**What it really means:** *the fraction of synthetic rows you would have to reach in and
change to make this column's histogram match reality.* TVD = 0.10 means 10% of your rows
have the wrong value in that column, in the aggregate sense. It is bounded in [0, 1].

**What it is blind to:** everything about *relationships*. A synthetic table that keeps
every column's histogram perfect but shuffles each column independently — destroying all
correlation — scores a near-perfect 1-way TVD. This is why it is never reported alone.

#### 2-way TVD  (`tvd_2way`) ↓ lower is better

Same computation, but on the *joint* histogram of every **pair** of columns, averaged over
all pairs.

**What it really means:** *do two columns co-occur in the synthetic data the way they do in
reality?* The independent-shuffle attack above fails this badly. This is the honest
single-number fidelity metric, and it is the one worth quoting.

**What it is blind to:** three-way and higher structure, and — critically — it treats all
pairs equally. The one pair you actually care about (a feature and the outcome) contributes
the same $1/\binom{n}{2}$ weight as any junk pair. Section 4.3 shows a case where 2-way TVD
looks fine and the feature→outcome relationship is *inverted*.

#### Average correlation difference  (`avg_correlation_diff`) ↓ lower is better

For each pair of columns, compute **Cramér's V** (a 0–1 measure of how strongly two
categorical variables are associated — think of it as correlation for categories, derived
from the chi-squared statistic and bias-corrected for table size). Take
$|V_{\text{real}} - V_{\text{synth}}|$, average over pairs.

**What it really means:** 2-way TVD asks *is the joint distribution shaped right*; this asks
*is the association as strong as it should be*. A generator that keeps relationships but
waters them down (a common GAN failure) is caught here more cleanly than by TVD.

### 2.2 Usefulness — "can anyone actually do work with this data?"

#### Downstream accuracy, TSTR  (`downstream_accuracy_mlp` / `_lr` / `_rf`) ↑ higher is better

**T**rain on **S**ynthetic, **T**est on **R**eal. Fit a classifier (MLP, logistic
regression, random forest) on the synthetic table to predict the outcome; evaluate its
accuracy on a **held-out 30% of the real data** that no generator ever saw.

**What it really means:** the whole promise of synthetic data in one number — *if I hand you
only this fake table, can you build a model that works on real people?*

**What it is blind to, and this matters a lot here:** raw accuracy has no scale. It must be
read against two reference lines:

| Reference | Adult | COMPAS | Meaning |
|---|---|---|---|
| **Majority baseline** | 0.747 | 0.548 | Accuracy of ignoring the data entirely and always guessing the most common class. **Anything at or below this learned nothing.** |
| **TRTR ceiling** | 0.822 | 0.667 | Train on *real* data instead. This is the best you could hope for. |

Adult's outcome is 75/25 imbalanced, so a method scoring 0.75 has achieved *exactly nothing*.
Throughout this notebook you will see two derived quantities instead of raw accuracy:

- **`lift`** = accuracy − majority baseline. **How much did the synthetic data actually
  teach the model?** Zero means nothing; negative means it taught the model something wrong.
- **`acc_retained`** = accuracy ÷ TRTR. What fraction of the real-data model's performance
  survived synthesis.

### 2.3 Privacy — "how exposed is any one person?"

#### ε (epsilon), with δ  (`epsilon`, `spent_epsilon`, `noise_multiplier`) ↓ lower is more private

**(ε, δ)-differential privacy.** Take any one person's record. Run the generator on the
dataset *with* them and *without* them. DP guarantees that the probability of producing any
particular synthetic output changes by at most a factor of $e^{\varepsilon}$ (with a
$\delta$ probability of the bound failing entirely; here δ = 1e-9).

**What it really means:** *an adversary who sees the synthetic data cannot confidently tell
whether you were in the input.* It is a worst-case guarantee — it holds against an attacker
who knows every other record in the dataset and has unlimited compute. That strength is why
it is the standard, and also why the numbers look harsh.

| ε | Read as |
|---|---|
| **1** | Strong. Genuinely defensible protection. |
| **10** | Moderate. Common in deployed systems; the guarantee is real but loose. |
| **1000** | Nominal. $e^{1000}$ is not a meaningful bound — this is essentially "the algorithm has the right *shape*, but is not protecting anyone." Included as the *no-privacy-cost* end of the curve. |

**The one thing people get wrong:** ε is not a probability and it is not linear. Going from
ε=1000 to ε=10 is a far smaller real-world change than going from ε=10 to ε=1.

**How it is spent here:** marginal methods (MST/PrivBayes/PrivSyn) spend the budget on the
noisy counts, tracked with zero-concentrated DP. The DP-GAN variants spend it on gradient
steps in the discriminator (DP-SGD: clip each per-sample gradient, add Gaussian noise),
tracked with an RDP/moments accountant. `noise_multiplier` is the resulting σ.

### 2.4 Fairness — "who does the model say yes to?"

All fairness metrics here are measured on the **predictions of a classifier that was trained
on the synthetic data**. That is the point: the intervention happens at generation time, and
we ask whether it propagates all the way to a deployed model's behaviour.

Every protected attribute is binarised as *most-common-value vs everyone else* (using the
real data's mode, so the comparison is identical across references). Where a run has several
protected attributes, we report the **worst case** over them — protecting more attributes
should not be rewarded with a better-looking average.

**A worked example used throughout this section.** A model is asked "will this person earn
over \$50k?" for **100 men and 100 women**. Here is what it does, and what really happened:

| | says YES | says NO | *truly* high earners | of those, model says YES |
|---|---|---|---|---|
| **Men** | 30 | 70 | 40 | 28 |
| **Women** | 15 | 85 | 25 | 15 |

Every metric below is a different question about this one table.

#### Demographic parity gap  (`fairness_gap`) ↓ lower is better

$$\big|\;\Pr(\hat{Y}{=}1 \mid S{=}1) - \Pr(\hat{Y}{=}1 \mid S{=}0)\;\big|$$

**What it really means:** *does the model say yes at the same rate to both groups?*

> **In the example:** it says yes to 30/100 men = 0.30 and 15/100 women = 0.15.
> **Gap = |0.30 − 0.15| = 0.15.**

It looks only at the model's output — it does not care whether the prediction was correct.
A gap of 0.15 means *"men are approved 15 percentage points more often than women."* That
unit — **percentage points of approval rate** — is what every `fairness_gap` number in this
notebook is in. 0.01 is negligible; 0.30 is enormous.

**What it is blind to — read this next to accuracy, always:** a model that predicts the same
class for **every single person** has a demographic parity gap of exactly 0. It is perfectly
"fair" and completely useless. Collapsed generators produce exactly this. Sections 7 and 8
flag these cases explicitly rather than letting them masquerade as wins.

#### Conditional demographic parity gap  (`cond_fairness_gap`) ↓ lower is better

The same gap, but computed *within* each combination of the **admissible** attributes, then
averaged weighted by group size.

**What it really means:** *compare people who are alike on the factors we've agreed are
legitimate.* If two people have the same education and work the same hours, does the model
still treat them differently by sex? This is the metric that encodes the belief that **some
disparity is explainable and some is not**.

> **In the example:** split those 200 people by education. Among *graduates* the model says
> yes to 50% of men and 48% of women (gap 0.02); among *non-graduates*, 12% and 10%
> (gap 0.02). Weighted together, **conditional gap ≈ 0.02** against a raw gap of 0.15.
> Reading: almost the entire raw disparity was *education*, not sex. Had the conditional gap
> come back at 0.14, the opposite reading — the model is separating men from women even among
> people with identical education.

**The interesting read:** the *difference* between `fairness_gap` and `cond_fairness_gap`
tells you how much of the raw disparity is routed through the admissible attributes. A big
raw gap that shrinks when conditioned = the disparity flows through education/hours. A raw
gap that survives conditioning = the model is using something it should not.

#### TPRB and TNRB  (`max_abs_tprb__real`, `max_abs_tnrb__real`) ↓ lower is better

- **TPRB** — among people whose *true* outcome is positive, is the model equally likely to
  correctly flag them across groups? $\Pr(\hat{Y}{=}1|S{=}1,Y{=}1) - \Pr(\hat{Y}{=}1|S{=}0,Y{=}1)$
- **TNRB** — the same for true negatives.

**What they really mean:** demographic parity asks about *rates*; these ask about *errors*.
On COMPAS: of the defendants who actually did reoffend, does the model catch Black and white
defendants equally often (TPRB)? And of those who did *not* reoffend, is it equally likely to
correctly clear them (TNRB)? Together they are "equalised odds".

**Why both exist:** they genuinely conflict. When base rates differ between groups, it is
mathematically impossible to equalise demographic parity and error rates at the same time
(the COMPAS impossibility result). You have to choose which one your application cares about —
so we measure all of them rather than pretending there is one right answer — and Section
7.6 shows what our mechanisms actually did to each.

### 2.4b Three different things in this notebook are called a "gap". Do not mix them up.

This trips everyone up, so it gets its own subsection. All three are in units of
*percentage points of approval rate*, but they answer completely different questions.

| | Name in tables | Question it answers | Good value |
|---|---|---|---|
| **1. Level** | `fairness_gap`, `baseline_gap`, `final_gap` | *How unfair is this model, right now?* | near 0 |
| **2. Change** | `gap_delta`, Δ gap | *How much did turning a mechanism on move it?* | very negative |
| **3. Measurement error** | `audit_error` | *Does the synthetic data tell the truth about its own gap?* | near 0 |

**1. The level (`fairness_gap`).** The 0.15 from the worked example above. A property of one
model: *men are approved 15 points more often than women.*

**2. The change (`gap_delta`).** Section 7.2 takes two runs identical in every way — same
dataset, generator, ε, role split, **and seed** — except one used a fairness mechanism and one
did not, then subtracts:

$$\texttt{gap\_delta} \;=\; \underbrace{\text{gap with the mechanism}}_{\text{e.g. }0.06} \;-\; \underbrace{\text{gap without it}}_{\text{e.g. }0.20} \;=\; -0.14$$

Negative = the mechanism helped. It is a *difference of two levels*, and that is precisely
why it is the careful version: everything that could confound the comparison is held fixed,
so seed noise cancels instead of being averaged over.

> ### ⚠️ The trap: a small `gap_delta` is not the same as a bad result.
>
> `gap_delta` measures **how far something moved**, not **where it ended up**. A generator
> that starts at a gap of 0.02 *cannot* produce a large negative delta — there is only 0.02
> of gap available to remove. It will look inert on a Δ chart while actually being the
> fairest thing in the table.
>
> This is exactly what happens to **MST on Adult** (Section 7.2), and it is why that section
> reports **three columns together — where it started, how far it moved, where it ended —**
> rather than the delta alone. Judge a *mechanism* by the delta; judge a *generator* by the
> final level. And check the accuracy lift before believing either, because a model that
> learned nothing scores a perfect 0.00 gap.

**3. The measurement error (`audit_error`).** Not about the model's fairness at all — about
whether the *synthetic dataset* reports its own fairness honestly. Defined in 2.5, measured
in 7.3, and it is the axis corresponding to DECAF's Definition 4 (Section 7.7).

### 2.5 The Distributional Fairness axis — "does the synthetic data tell the truth about its own fairness?"

This one is subtle and is worth the paragraph.

Take the classifier trained on synthetic data. You can score its fairness against **two
different populations**:

- **real reference** (`max_abs_dp_gap__real`, and the plain `fairness_gap`) — score it on the
  held-out real people. *This is ground truth: how the model will actually behave in the world.*
- **synthetic reference** (`synthref__*`, `max_abs_dp_gap__synth`) — score it on the synthetic
  data itself. *This is what an analyst would measure if the real data were locked away and
  all they had was the synthetic release.*

**What it really means:** the gap between the two is an **audit error**. The whole premise of
releasing synthetic data is that people can do their analysis on it instead of the real thing.
If someone audits a model for bias using only the synthetic release and gets a materially
different answer than they would have gotten from real data, the release is *actively
misleading* — arguably worse than no release. Section 7.3 measures this.

### 2.6 Run status

- **`done`** — everything computed.
- **`partial` / `collapsed`** — the generator produced a synthetic outcome column with only
  **one** value (e.g. every fake person is labelled 'did not reoffend'). No classifier can be
  fit, so all downstream utility and fairness metrics are undefined and left blank. Fidelity
  metrics still apply. **Collapse rate is itself a headline result** for the DP-GAN family
  (Section 8) — and note that these runs are *excluded* from fairness averages, so they
  cannot fake a good fairness score.

In [ ]:
# Quick-reference glossary you can scroll back to
glossary = pd.DataFrame([
 ('tvd_1way',           'Fidelity',  'lower', '% of rows with a wrong value, per column, averaged', 'blind to all relationships'),
 ('tvd_2way',           'Fidelity',  'lower', 'same, for every PAIR of columns', 'weights the outcome pair like any other'),
 ('avg_correlation_diff','Fidelity', 'lower', 'is each association as STRONG as in reality', 'ignores direction'),
 ('acc (mlp/lr/rf)',    'Usefulness','higher','train on synthetic, test on real holdout', 'meaningless without a baseline'),
 ('lift',               'Usefulness','higher','accuracy MINUS always-guess-majority', 'the honest version of accuracy'),
 ('acc_retained',       'Usefulness','higher','fraction of train-on-real accuracy kept', '-'),
 ('epsilon',            'Privacy',   'lower', 'worst-case exp(eps) bound on one persons influence', 'not a probability; not linear'),
 ('fairness_gap',       'Fairness',  'lower', 'demographic parity: |P(yes|A) - P(yes|B)|', 'a constant predictor scores 0'),
 ('cond_fairness_gap',  'Fairness',  'lower', 'same, among people alike on admissible attrs', '-'),
 ('max_abs_tprb__real', 'Fairness',  'lower', 'equal catch-rate among the truly positive', 'conflicts with parity by construction'),
 ('max_abs_tnrb__real', 'Fairness',  'lower', 'equal clear-rate among the truly negative', 'conflicts with parity by construction'),
 ('synthref__*',        'DF axis',   '-',     'same metric scored on synthetic instead of real', 'gap vs real = audit error'),
 ('collapsed',          'Validity',  'lower', 'synthetic outcome had only one class', 'these rows are dropped from fairness stats'),
], columns=['metric', 'family', 'better', 'what it measures', 'what it misses'])
glossary.style.hide(axis='index')

## 3. Who counts as protected — the attribute role splits

Causal fairness needs you to sort the columns into three roles. **This is a modelling
choice, not a fact about the data**, so we ran three different splits per dataset to see how
much the conclusions depend on it.

| Role | Meaning |
|---|---|
| **Protected (sensitive)** | The attribute whose influence on the outcome we consider illegitimate. |
| **Admissible** | Attributes through which disparity is considered *acceptable* — a legitimate business justification. Education affecting income is fine; that education itself differs by group is a separate problem. |
| **Outcome** | What the downstream model predicts. |

The `prefair` split for each dataset is taken from the PreFair paper's Table 1, so those rows
are directly comparable to published numbers. The others deliberately stress the extremes:
one protects a single attribute with a *broad* set of excuses, the other protects several
attributes with a *narrow* set.

**Why this axis matters:** the CF mechanism only blocks pathways that do **not** pass through
an admissible attribute. Widen the admissible set and you are declaring more of the disparity
legitimate — CF should then do less work. That prediction is tested in Section 7.5.

In [ ]:
roles = (runs[['dataset', 'role_config', 'protected_attrs', 'admissible_attrs', 'outcome_attr']]
         .drop_duplicates().sort_values(['dataset', 'role_config']).reset_index(drop=True))
roles['n_protected'] = roles.protected_attrs.str.count(',') + 1
roles['n_admissible'] = roles.admissible_attrs.str.count(',') + 1
roles.style.hide(axis='index')

### The four fairness mechanisms

All four operate on the **causal graph used to generate the data**, not on the model. This is
the leverage that only synthetic data gives you.

| Mechanism | What it removes | The intuition |
|---|---|---|
| **none** | nothing | Baseline. Generate from the full graph, disparities and all. |
| **FTU** *(Fairness Through Unawareness)* | the **direct** edge protected → outcome | "Don't let sex point straight at income." Cheap and weak: everything can still flow through proxies. |
| **DP** *(Demographic Parity — not privacy!)* | **every** directed path protected → outcome | The strongest cut. Nothing about the protected attribute reaches the outcome, legitimate or not. |
| **CF** *(Conditional Fairness)* | only paths that do **not** pass through an admissible attribute | The nuanced one. Disparity flowing through education is allowed to survive; disparity flowing through anything else is cut. |

For the GAN-based methods the cut is applied at *generation* time: the listed parent columns
are randomly permuted when sampling that variable, so the child is drawn as if its parent
carried no information (DECAF's "surrogate value substitution"). For the marginal-based
methods it constrains which marginals are eligible to be selected in the first place.

**Expected ordering, if the theory is right:** `none` ≥ `FTU` ≥ `CF` ≥ `DP` on the fairness
gap, with cost to utility increasing in the same direction.

## 4. Data quality — does the synthetic data look like the real data?

Fidelity first, because nothing downstream means anything if the table is garbage.

### 4.1 The fidelity leaderboard

In [ ]:
fid = (runs.groupby(['dataset', 'sdg_method'])
           .agg(tvd_1way=('tvd_1way', 'mean'), tvd_2way=('tvd_2way', 'mean'),
                corr_diff=('avg_correlation_diff', 'mean'), n=('run_id', 'size'))
           .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ds in zip(axes, DATASETS):
    d = fid[fid.dataset == ds].sort_values('tvd_2way', ascending=True)
    y = np.arange(len(d))
    ax.barh(y, d.tvd_2way, color=[COLORS[m] for m in d.sdg_method], height=0.62)
    ax.set_yticks(y); ax.set_yticklabels([nice(m) for m in d.sdg_method])
    ax.invert_yaxis()
    ax.set_xlabel('2-way TVD  (lower = better)')
    ax.set_title('%s — pairwise fidelity' % nice(ds))
    ax.set_xlim(0, max(d.tvd_2way) * 1.22)
    for yi, v in zip(y, d.tvd_2way):
        ax.text(v + max(d.tvd_2way) * 0.015, yi, '%.3f' % v, va='center', fontsize=10)
fig.suptitle('Marginal-based methods dominate fidelity; causal GANs are 5-20x worse',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

fid.pivot(index='sdg_method', columns='dataset',
          values=['tvd_1way', 'tvd_2way', 'corr_diff']).round(4)

**Read:** MST, PrivSyn and PrivBayes are in a different league — MST reproduces pairwise
structure to within 3-5% on both datasets. The causal-GAN family pays a large fidelity tax
for its structural constraint. That is the expected shape: marginal methods optimise exactly
this quantity, GANs do not.

The interesting row is **DECAF+CTGAN on Adult: 2-way TVD 0.179 against plain DECAF's
0.543** — a 3× improvement from swapping the data representation alone, with no change to
the causal machinery. On COMPAS the ranking goes the other way (plain DECAF 0.077 vs
DECAF+CTGAN 0.124), and this reversal is *itself* a result of the epoch correction: in the
superseded batch COMPAS looked like the representation win and Adult like the failure. Once
both were trained to their own best settings, the win moved. Representation quality and
training length are not separable knobs. More on that in Sections 4.3c and 8.

### 4.2 What the privacy budget costs in fidelity

In [ ]:
priv = runs[runs.epsilon.notna()]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for ax, ds in zip(axes, DATASETS):
    d = priv[priv.dataset == ds]
    for m, sub in d.groupby('sdg_method'):
        s = sub.groupby('epsilon').tvd_2way.agg(['mean', 'std', 'size'])
        ci = 1.96 * s['std'] / np.sqrt(s['size'])
        ax.errorbar(s.index, s['mean'], yerr=ci, marker='o', capsize=3,
                    label=nice(m), color=COLORS[m], lw=2)
    ax.set_xscale('log'); ax.set_xticks([1, 10, 1000])
    ax.set_xticklabels(['1\n(strong)', '10\n(moderate)', '1000\n(nominal)'])
    ax.set_xlabel('privacy budget  \u03b5'); ax.set_ylabel('2-way TVD (lower = better)')
    ax.set_title(nice(ds))
    ax.legend(fontsize=9, frameon=False)
fig.suptitle('Cost of privacy in fidelity: flat for marginal methods, chaotic for GANs',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

**Read:** For MST / PrivBayes / PrivSyn the curves are almost flat — going from a nominal
ε=1000 to a genuinely strong ε=1 costs very little fidelity on these datasets. That is the
good news that makes the rest of the study viable: **you can have real privacy nearly for
free here.**

The GAN curves are not just worse, they are *non-monotone* — DECAF+DP-GAN on Adult gets
**better** as ε tightens. That is not privacy helping. It is a weight-clipped critic being
unstable, with the DP noise acting as accidental regularisation. Treat those rows as
"this architecture does not work on this dataset", not as a tuned result.

### 4.3 ⚠️ When good fidelity lies

This is worth its own figure, because it is the most important methodological lesson in the
batch and it would be invisible from the summary tables.

> **Note:** this section diagnoses the *superseded* batch — the rows the setup cell filters
> out. The bug described here has since been fixed and the grid re-run (Section 4.3c), so the
> tables above no longer show it. It is kept in full because the failure mode is general and
> the diagnosis is the transferable part.

DECAF+CTGAN on Adult had a perfectly respectable 2-way TVD of ~0.24 on **every one of the 5
seeds**. But the downstream accuracy is bimodal: three seeds land near 0.74, two seeds land
near **0.32** — far *below* the 0.747 you would get by guessing. An accuracy that far below
chance means the model learned a relationship that is **inverted**: the synthetic data taught
it that the features predicting high income predict low income.

The marginals were fine. The pairwise TVD was fine. The joint structure connecting features
to the outcome was backwards, and no fidelity metric in the standard toolkit noticed.

In [ ]:
# NOTE: this cell reads `superseded` -- the OLD batch -- on purpose. It is the
# evidence for the bug. The fixed version of this same cell is Section 4.3c.
d = superseded[(superseded.dataset == 'adult') & (superseded.sdg_method == 'decaf_ctgan')]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
for seed, sub in d.groupby('seed'):
    ax.scatter(sub.tvd_2way, sub.acc, s=45, alpha=0.8, label='seed %d' % seed)
ax.axhline(MAJORITY['adult'], color='k', ls='--', lw=1.4)
ax.text(0.30, MAJORITY['adult'] + 0.012, 'always-guess-majority (0.747)', fontsize=9)
ax.axhline(0.5, color='crimson', ls=':', lw=1.4)
ax.text(0.30, 0.515, 'coin flip', fontsize=9, color='crimson')
ax.set_xlabel('2-way TVD (fidelity)'); ax.set_ylabel('downstream accuracy')
ax.set_title('Same fidelity, two completely different outcomes')
ax.legend(fontsize=9, frameon=False, ncol=2)

ax = axes[1]
s = d.groupby('seed').acc.mean()
ax.bar([str(i) for i in s.index], s.values,
       color=['#55A868' if v > 0.6 else '#C44E52' for v in s.values])
ax.axhline(MAJORITY['adult'], color='k', ls='--', lw=1.4)
ax.set_xlabel('seed'); ax.set_ylabel('downstream accuracy')
ax.set_title('Seeds 1 and 3 learned an INVERTED relationship')
for i, v in enumerate(s.values):
    ax.text(i, v + 0.012, '%.3f' % v, ha='center', fontsize=10)
plt.tight_layout(); plt.show()

print('SUPERSEDED batch -- per-seed spread, Adult / DECAF+CTGAN (12 cells each):')
display(d.groupby('seed')[['tvd_1way', 'tvd_2way', 'acc', 'downstream_accuracy_lr',
                           'downstream_accuracy_rf']].mean().round(4))

fixed = runs[(runs.dataset == 'adult') & (runs.sdg_method == 'decaf_ctgan')]
print('\nSame cell AFTER the fix (600 epochs) -- the bimodality is gone:')
fixed.groupby('seed')[['tvd_1way', 'tvd_2way', 'acc']].mean().round(4)

**Why this matters for the paper:** it is a concrete argument that the field's standard
fidelity metrics are insufficient for evaluating *causally structured* generators, and it
motivates reporting **lift over the majority baseline across multiple seeds** as a minimum
validity check.

#### 4.3b Root cause, confirmed: we selected the model on the wrong metric

A follow-up sweep re-trained Adult / DECAF+CTGAN at 30 / 60 / 120 / 300 epochs across 3
seeds each (12 fits). It identifies the cause exactly, and it is a *selection* error rather
than a bug in the generator.

The batch used **30 epochs**, chosen during tuning by minimising **1-way TVD** on a single
seed. That was the wrong criterion twice over:

1. **1-way TVD is non-monotone in training and essentially uninformative here** — it goes
   0.105 → 0.112 → 0.091 → 0.100 across the four settings. It cannot rank them.
2. **2-way TVD — the metric that can actually see relationships — improves with training**,
   0.215 → 0.217 → 0.182 → 0.181. It says train *longer*, the opposite of what we did.

Single-seed tuning compounded it: the 1-way TVD of 0.074 that won at 30 epochs was a lucky
draw, and re-running that same configuration across seeds gives 0.093–0.118.

**At 300 epochs the pathology is gone**: mean accuracy 0.771, above the 0.747 baseline for
the first time, with no inverted seed and the synthetic positive rate averaging 0.237 against
a true 0.247. **Under-training, selected for by the wrong metric, was the whole problem.**

In [ ]:
sweep = pd.read_csv(DATA / 'adult_ctgan_epoch_sweep.csv')
s = sweep.groupby('epochs').agg(
    tvd_1way=('tvd1', 'mean'), tvd_2way=('tvd2', 'mean'),
    pos_rate=('pos_rate', 'mean'), pos_rate_min=('pos_rate', 'min'),
    pos_rate_max=('pos_rate', 'max'), acc=('acc_mlp', 'mean'),
    acc_worst=('acc_mlp', 'min'), fit_secs=('secs', 'mean'))
s['lift'] = s.acc - MAJORITY['adult']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
ax = axes[0]
ax.plot(s.index, s.tvd_1way, 'o-', lw=2, color='#DD8452', label='1-way TVD (used for tuning)')
ax.plot(s.index, s.tvd_2way, 's-', lw=2, color='#4C72B0', label='2-way TVD (sees relationships)')
ax.axvline(30, color='crimson', ls=':', lw=2)
ax.text(33, ax.get_ylim()[1] * 0.93, 'what the\nbatch used', color='crimson', fontsize=9)
ax.set_xscale('log'); ax.set_xticks(s.index); ax.set_xticklabels(s.index)
ax.set_xlabel('training epochs'); ax.set_ylabel('TVD (lower = better)')
ax.set_title('The two fidelity metrics disagree')
ax.legend(fontsize=8.5, frameon=False)

ax = axes[1]
ax.fill_between(s.index, s.pos_rate_min, s.pos_rate_max, alpha=0.25, color='#8172B3')
ax.plot(s.index, s.pos_rate, 'o-', lw=2, color='#8172B3', label='synthetic (band = seed range)')
ax.axhline(sweep.real_pos.mean(), color='k', ls='--', lw=1.5)
ax.text(60, sweep.real_pos.mean() + 0.012, 'true rate (0.247)', fontsize=9)
ax.set_xscale('log'); ax.set_xticks(s.index); ax.set_xticklabels(s.index)
ax.set_xlabel('training epochs'); ax.set_ylabel('P(income > 50k) in synthetic data')
ax.set_title('The outcome marginal converges')
ax.legend(fontsize=8.5, frameon=False)

ax = axes[2]
ax.plot(s.index, s.acc, 'o-', lw=2, color='#55A868', label='mean over 3 seeds')
ax.plot(s.index, s.acc_worst, 'v--', lw=1.5, color='#C44E52', label='worst seed')
ax.axhline(MAJORITY['adult'], color='k', ls='--', lw=1.5)
ax.text(60, MAJORITY['adult'] - 0.018, 'guess majority (0.747)', fontsize=9)
ax.set_xscale('log'); ax.set_xticks(s.index); ax.set_xticklabels(s.index)
ax.set_xlabel('training epochs'); ax.set_ylabel('downstream accuracy')
ax.set_title('Only 300 epochs clears the baseline')
ax.legend(fontsize=8.5, frameon=False)
fig.suptitle('Adult / DECAF+CTGAN: the batch trained 10x too little, because 1-way TVD said so',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()
s.round(4)

**Read, and the recommendation for the re-run:**

| epochs | 2-way TVD | accuracy | lift | worst seed | fit cost |
|---|---|---|---|---|---|
| 30 *(used)* | 0.215 | 0.720 | **−0.027** | 0.670 | 175 s |
| 60 | 0.217 | 0.750 | +0.003 | 0.737 | 360 s |
| 120 | 0.182 | 0.756 | +0.009 | **0.751** | 730 s |
| **300** | **0.181** | **0.771** | **+0.024** | 0.746 | 1830 s |

**300 epochs** is the pick: it is the only setting where DECAF+CTGAN beats the trivial
predictor on Adult, and 2-way TVD has plateaued by then. **120 epochs** is the value option —
2.5× cheaper, statistically indistinguishable fidelity, and the *tightest* seed spread, at
the cost of most of the lift.

**Honest caveat:** the seed spread in the outcome marginal narrows but does not close
(0.14–0.33 even at 300 epochs). Longer training fixes the *inversion* pathology; it does not
make this generator stable. Multi-seed reporting stays mandatory.

**Scope:** this sweep covers **Adult / DECAF+CTGAN only**, and it was run on CPU. Section
4.3c extends it to all eight (method × dataset) combinations — and overturns part of the
conclusion above.

### 4.3c The full sweep: every GAN cell, and why "300 epochs" is not the answer

Section 4.3b characterised one cell. But **every** GAN cell in the grid was tuned the same
wrong way — single-seed 1-way TVD — so none of them could be trusted either. This sweep
covers all eight (method × dataset) combinations, 3 seeds each, and for the two DP variants
it sweeps **epochs and ε jointly**: under DP-SGD each extra epoch is extra steps to pay for,
so at a fixed budget more training means more noise per step. "Train longer" is free for
plain CTGAN and is not free under DP. **114 fits, 0 failures.**

**Selection rule**, applied in this order — the rule is the point, more than any single
number: (1) never collapse to a single class; (2) *every* seed must beat the trivial
majority predictor; (3) among what survives, lowest 2-way TVD. Rule 2 is what 4.3b taught
us: a setting can win on fidelity while producing a model that has not learned the outcome.

In [ ]:
sweep = pd.read_csv(DATA / 'gan_epoch_sweep_all.csv')
MAJ = {d: MAJORITY[d] for d in MAJORITY}
sweep['lift'] = sweep.acc_mlp - sweep.dataset.map(MAJ)

agg = (sweep.groupby(['method', 'dataset', 'epsilon', 'epochs'], dropna=False)
        .agg(tvd2=('tvd2', 'mean'), lift=('lift', 'mean'),
             lift_min=('lift', 'min'), collapse=('collapsed', 'mean'),
             mins=('secs', lambda s: s.mean() / 60))
        .reset_index())

OLD = {('decaf', 'adult'): 30, ('decaf', 'compas'): 200,
       ('decaf_ctgan', 'adult'): 30, ('decaf_ctgan', 'compas'): 20,
       ('decaf_dpgan', 'adult'): 60, ('decaf_dpgan', 'compas'): 100,
       ('decaf_dpctgan', 'adult'): 60, ('decaf_dpctgan', 'compas'): 200}

picks = []
for (m, ds, eps), sub in agg.groupby(['method', 'dataset', 'epsilon'], dropna=False):
    ok, rule = sub[(sub.collapse == 0) & (sub.lift_min > 0)], 'all seeds beat baseline'
    if ok.empty:
        ok, rule = sub[(sub.collapse == 0) & (sub.lift > 0)], 'beats baseline on average'
    if ok.empty:
        ok, rule = sub[sub.collapse == 0], 'NEVER beats baseline'
    if ok.empty:
        ok, rule = sub, 'COLLAPSES at every setting'
    b = ok.loc[ok.tvd2.idxmin()]
    picks.append(dict(method=m, dataset=ds, epsilon=eps, was=OLD[(m, ds)],
                      picked=int(b.epochs), rule=rule, tvd2=b.tvd2,
                      lift=b.lift, lift_min=b.lift_min, mins=b.mins))
picks = pd.DataFrame(picks)
picks['changed'] = np.where(picks.picked != picks.was, 'CHANGED', '')
display(picks.round(3).style.hide(axis='index')
        .background_gradient(subset=['lift'], cmap='RdYlGn'))

fig, axes = plt.subplots(2, 4, figsize=(16, 7.5), sharex=False)
for ax, ((m, ds), sub) in zip(axes.ravel(), agg.groupby(['method', 'dataset'])):
    for eps, e in sub.groupby('epsilon', dropna=False):
        e = e.sort_values('epochs')
        lab = 'no DP' if pd.isna(eps) else ('ε=%g' % eps)
        ax.plot(e.epochs, e.lift, 'o-', label=lab, lw=1.8, ms=5)
    ax.axhline(0, color='crimson', lw=1.3, ls='--')
    ax.axvline(OLD[(m, ds)], color='#666666', lw=1.1, ls=':')
    ax.set_xscale('log')
    ax.set_title('%s \u2014 %s' % (nice(m), nice(ds)), fontsize=10)
    ax.set_xlabel('epochs (log)'); ax.set_ylabel('accuracy lift vs trivial')
    ax.legend(fontsize=7.5, frameon=False)
for ax in axes.ravel()[agg.groupby(['method', 'dataset']).ngroups:]:
    ax.set_visible(False)
fig.suptitle('Red line = the trivial predictor. Dotted grey = the epoch count the '
             'original batch used.\nBelow red means the generator never learned the outcome.',
             fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Read — three findings, and the first one corrects Section 4.3b.**

**1. 300 epochs was not stable; 600 is.** Re-running Adult/DECAF+CTGAN at 300 epochs on GPU
instead of CPU — *same three seeds, same code, only the RNG stream differs* — produced a
catastrophic failure the CPU run never showed: seed 0 came back with a synthetic positive
rate of **0.83 against a true 0.247**, and accuracy **0.282**. The mean lift at 300 epochs is
−0.134. At **600 epochs all three seeds land at 0.18–0.23** with accuracy 0.784–0.801. So
4.3b's headline number was itself a lucky draw. The deeper lesson survives intact and is
actually strengthened: **three seeds on one device was still not enough evidence**, and the
instability is a property of the generator that more training — not a better metric — fixes.

**2. There is no global epoch count. The direction of the effect flips by dataset.**
On Adult, CTGAN needs far *more* training than the batch gave it (30 → 600). On COMPAS,
fidelity *degrades* monotonically with training — the best 2-way TVD is at 20 epochs, the
value already in use. But 20 epochs scores **below the majority baseline** (lift −0.006),
and 500 epochs reaches **+0.085** at worse TVD. Same trap as Adult, opposite direction:
selecting on fidelity picks a model that has not learned the outcome.

**3. Under DP the optimum moves with the privacy budget — so they had to be swept together.**
DECAF+DP-CTGAN on COMPAS wants 400 epochs at ε=1000 (lift +0.084) but 200 at ε=1, where
nothing clears the baseline at all. DECAF+DP-GAN on COMPAS **collapses at every epoch count
tried at ε=1** (100/200/500), and on Adult collapses in 2 of 3 seeds at 60 and 300 epochs,
leaving 120 as the only survivable setting. Had we swept epochs at one ε and reused the
answer, we would have repeated the original mistake in a new dimension.

#### What actually changed

| Method | Dataset | was | now | why |
|---|---|---|---|---|
| DECAF+CTGAN | Adult | 30 | **600** | 30 never beat baseline; 300 unstable |
| DECAF+CTGAN | COMPAS | 20 | **500** | 20 scored below the trivial predictor |
| DECAF | Adult | 30 | **300** | fidelity 0.707 → 0.548; still never beats baseline |
| DECAF | COMPAS | 200 | **1000** | fidelity 0.330 → 0.091, no collapse |
| DECAF+DP-GAN | Adult | 60 | **120** | only count that avoids collapse at ε=1000 |
| DECAF+DP-GAN | COMPAS | 100 | 100 | already correct |
| DECAF+DP-CTGAN | Adult | 60 | 60 | never clears baseline at any count |
| DECAF+DP-CTGAN | COMPAS | 200 | **400** | lift +0.023 → +0.084 |

Six of eight cells changed. The re-run under these settings is batch
`gan-retuned-2026-08-04` (960 runs, 0 failed on Adult, 1 failed on COMPAS). **It has landed,
and Sections 4–8 below are cut against it.** The superseded rows are still in the data file
so the correction can be shown directly:

In [ ]:
# Before/after: the same grid cells, old epoch settings vs swept ones.
GANS = ['decaf', 'decaf_ctgan', 'decaf_dpgan', 'decaf_dpctgan']

def _summary(df):
    return df.groupby(['dataset', 'sdg_method']).agg(
        collapse=('collapsed', 'mean'),
        tvd_1way=('tvd_1way', 'mean'),
        acc=('acc', 'mean'),
        lift=('lift', 'mean'),
        gap=('fairness_gap', 'mean'))

before = _summary(superseded[superseded.sdg_method.isin(GANS)])
after = _summary(runs[runs.sdg_method.isin(GANS)])
cmp = before.join(after, lsuffix='_before', rsuffix='_after')
cmp = cmp[sorted(cmp.columns, key=lambda c: (c.rsplit('_', 1)[0], c.endswith('_before') == False))]
display(cmp.round(4))

print('\nLift = accuracy minus the trivial majority predictor. Negative = worse than guessing.')
print('Sign flips on lift (the cells that went from useless to useful):')
flip = cmp[(cmp.lift_before < 0) & (cmp.lift_after > 0)]
display(flip[['lift_before', 'lift_after', 'acc_before', 'acc_after']].round(4))

**Read — the correction did what it was supposed to, and exposed a second problem.**

**Adult/DECAF+CTGAN was the worst cell in the project and is now a working one.** Accuracy
went **0.569 → 0.766** against a majority baseline of 0.747 — from 18 points *below* the
trivial predictor to 2 points above it. Fidelity improved at the same time (1-way TVD
0.118 → 0.101). Nothing was traded away; the model was simply undertrained.

**Plain DECAF improved on both datasets.** Adult 1-way TVD 0.457 → 0.338 and COMPAS
0.131 → **0.043**, the best fidelity any DECAF variant reaches here. So the mistuning was
never specific to the CTGAN backbone — it was in every GAN cell, exactly as the sweep
predicted.

**But look at the COMPAS fairness gaps: they went *up*, a lot.** DECAF+CTGAN's mean gap
went 0.075 → **0.270**, and plain DECAF's 0.175 → 0.229. This is not a regression. It is the
trivially-fair failure mode from Section 2.4 resolving itself: the old COMPAS models had
lift near zero — they had not learned the outcome, so they had no gap to show. A generator
that produces noise is perfectly fair and perfectly useless. Now that these models actually
predict something (lift +0.060), they reproduce the bias that is genuinely in COMPAS, and
the fairness mechanisms have something real to remove. **The higher gaps are the honest
ones.** This is the single strongest argument in the notebook for why fairness numbers must
always be read next to a lift column.

## 5. Privacy

The interesting question is not "does privacy cost something" (it does) but **"does privacy
interact with fairness?"** A common intuition says it must: DP noise hits small groups
hardest, so tightening ε should hurt minority groups and widen fairness gaps.

### 5.1 The privacy–utility–fairness triple

In [ ]:
pm = ['mst', 'privbayes', 'privsyn']  # methods with a working epsilon sweep
d = runs[(runs.sdg_method.isin(pm)) & runs.epsilon.notna() & runs.acc.notna()]

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
panels = [('tvd_2way', '2-way TVD', 'lower better'),
          ('lift', 'accuracy lift over majority', 'higher better'),
          ('fairness_gap', 'demographic parity gap', 'lower better')]
for r, ds in enumerate(DATASETS):
    for c, (col, lab, arrow) in enumerate(panels):
        ax = axes[r, c]
        for m, sub in d[d.dataset == ds].groupby('sdg_method'):
            s = sub.groupby('epsilon')[col].agg(['mean', 'std', 'size'])
            ci = 1.96 * s['std'] / np.sqrt(s['size'])
            ax.errorbar(s.index, s['mean'], yerr=ci, marker='o', capsize=3,
                        color=COLORS[m], label=nice(m), lw=2)
        ax.set_xscale('log'); ax.set_xticks([1, 10, 1000])
        if r == 1:
            ax.set_xlabel('privacy budget \u03b5')
        ax.set_ylabel('%s\n(%s)' % (lab, arrow) if c == 0 else lab, fontsize=10)
        ax.set_title('%s — %s' % (nice(ds), lab), fontsize=11)
        if r == 0 and c == 0:
            ax.legend(fontsize=9, frameon=False)
fig.suptitle('Tightening privacy barely moves fairness',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

d.groupby(['dataset', 'sdg_method', 'epsilon'])[
    ['tvd_2way', 'acc', 'lift', 'fairness_gap', 'cond_fairness_gap']].mean().round(4)

**Read:** the right-hand column is close to flat. Across a 1000× change in the privacy
budget, the demographic parity gap moves far less than the mechanism choice moves it
(Section 7). **Privacy and fairness behave like near-independent knobs on these datasets**,
which is a genuinely useful negative result — it means a practitioner does not have to trade
one against the other, and it contradicts the folk intuition.

*Caveat worth stating in any writeup:* this is two datasets with moderately sized protected
groups. The DP-hurts-minorities effect is real in the literature and would likely appear with
smaller subgroups. What we can say is that it is not the dominant effect at this scale.

### 5.2 Was the budget actually spent as requested?

In [ ]:
acct = runs[runs.spent_epsilon.notna()]
if len(acct):
    t = (acct.groupby(['dataset', 'sdg_method', 'epsilon'])
             .agg(requested=('epsilon', 'mean'), spent=('spent_epsilon', 'mean'),
                  noise_sigma=('noise_multiplier', 'mean'),
                  dp_steps=('dp_steps', 'mean'), n=('run_id', 'size'))
             .round(4))
    display(t)
    print('Sanity: every run spent <= its requested budget:',
          bool((acct.spent_epsilon <= acct.epsilon + 1e-6).all()))
else:
    print('no accountant traces recorded')

## 6. Usefulness — is a model trained on this data worth anything?

Reminder from Section 2.2: **raw accuracy is misleading on Adult**, whose outcome is 75/25.
The dashed lines are the two reference points that make the numbers mean something.

> **What is held fixed here.** Every figure in this section fixes **`mechanism = none`** — the
> un-intervened generator, which is the right comparison for "how good is this generator" —
> and shows **each ε separately** rather than averaging over them. Averaging across ε or
> mechanism is not safe for the causal-GAN variants (it hides ~10 accuracy points), and it is
> actively misleading for DECAF+DP-GAN, whose collapsed runs are dropped unevenly across ε.
> The cell at the end of this section quantifies both effects so you can check the claim.
> Each bar still averages over the 3 role splits and 5 seeds, where the spread is small.

In [ ]:
# HOLD THE OTHER AXES FIXED. Pooling over epsilon and mechanism is not safe here:
# it hides ~10 accuracy points on the GAN variants, and because collapsed runs are
# dropped, DP-GAN's surviving runs are unbalanced across epsilon in OPPOSITE
# directions on the two datasets (Adult 56/36/24, COMPAS 12/24/60 for eps 1/10/1000).
# So: fix mechanism='none' (the un-intervened generator) and show each epsilon separately.
u = runs[runs.acc.notna()]
base = u[u.fairness_mechanism == 'none']

agg = (base.groupby(['dataset', 'sdg_method', 'eps_label'])
           .acc.agg(['mean', 'std', 'size']).reset_index())
agg['sdg_method'] = pd.Categorical(agg.sdg_method, METHOD_ORDER, ordered=True)

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4))
EPS_HATCH = {'1': '///', '10': '\\\\\\', '1000': '', 'none': ''}
EPS_ALPHA = {'1': 0.45, '10': 0.7, '1000': 1.0, 'none': 1.0}
for ax, ds in zip(axes, DATASETS):
    d = agg[agg.dataset == ds].sort_values(['sdg_method', 'eps_label'])
    meths = [m for m in METHOD_ORDER if m in set(d.sdg_method)]
    xt, xl = [], []
    pos = 0.0
    for m in meths:
        sub = d[d.sdg_method == m]
        sub = sub.set_index('eps_label').reindex(
            [e for e in EPS_ORDER if e in set(sub.eps_label)]).reset_index()
        start = pos
        for _, r in sub.iterrows():
            ci = 1.96 * r['std'] / np.sqrt(r['size']) if r['size'] > 1 else 0
            ax.bar(pos, r['mean'], width=0.82, yerr=ci, capsize=2.5,
                   color=COLORS[m], alpha=EPS_ALPHA[r.eps_label],
                   hatch=EPS_HATCH[r.eps_label], edgecolor='white', linewidth=0.4)
            if r.eps_label != 'none':
                ax.text(pos, 0.012, '\u03b5=%s' % r.eps_label, ha='center',
                        va='bottom', fontsize=7, rotation=90, color='white')
            pos += 1
        xt.append((start + pos - 1) / 2); xl.append(nice(m))
        pos += 0.7
    d = agg[agg.dataset == ds]
    x = np.arange(0)  # reference lines below use axis coords
    ax.axhline(MAJORITY[ds], color='k', ls='--', lw=1.5)
    ax.axhline(TRTR[ds], color='seagreen', ls='-.', lw=1.5)
    ax.text(pos - 0.9, MAJORITY[ds] + 0.006, 'guess majority (%.3f)' % MAJORITY[ds],
            ha='right', fontsize=9)
    ax.text(pos - 0.9, TRTR[ds] + 0.006, 'train on REAL data (%.3f)' % TRTR[ds],
            ha='right', fontsize=9, color='seagreen')
    ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('downstream accuracy (train synth \u2192 test real)')
    ax.set_title('%s  —  mechanism = none' % nice(ds))
    ax.set_ylim(0, max(TRTR[ds], agg[agg.dataset == ds]['mean'].max()) * 1.18)
fig.suptitle('Anything below the black line learned nothing from the data'
             '   (bars split by \u03b5; paler = tighter privacy)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# The honest table: how much did the synthetic data actually teach a model?
# Broken out by epsilon, mechanism held at 'none'. `n` is the number of SURVIVING
# runs -- where it is below 15 the generator collapsed in the rest, and the mean is
# taken over the lucky ones only. Read those rows with suspicion.
tbl = (base.groupby(['dataset', 'sdg_method', 'eps_label'])
        .agg(accuracy=('acc', 'mean'), lift_over_majority=('lift', 'mean'),
             pct_of_real=('acc_retained', 'mean'), n=('run_id', 'size'))
        .reset_index())
expected = base.groupby(['dataset', 'sdg_method', 'eps_label']).size().rename('x')
allruns = (runs[runs.fairness_mechanism == 'none']
           .groupby(['dataset', 'sdg_method', 'eps_label']).size().rename('attempted'))
tbl = tbl.merge(allruns.reset_index(), on=['dataset', 'sdg_method', 'eps_label'])
tbl['survived'] = (tbl.n / tbl.attempted).round(2)
tbl['verdict'] = np.where(tbl.lift_over_majority > 0.05, 'genuinely useful',
                  np.where(tbl.lift_over_majority > 0.01, 'marginal',
                  np.where(tbl.lift_over_majority > -0.01, 'no better than guessing',
                           'WORSE than guessing')))
tbl.loc[tbl.survived < 1.0, 'verdict'] += '  (survivors only)'
tbl['sdg_method'] = pd.Categorical(tbl.sdg_method, METHOD_ORDER, ordered=True)
tbl['eps_label'] = pd.Categorical(tbl.eps_label, EPS_ORDER, ordered=True)
tbl = tbl.sort_values(['dataset', 'sdg_method', 'eps_label'])
tbl.round(4).style.hide(axis='index').background_gradient(
    subset=['lift_over_majority'], cmap='RdYlGn')

#### Does it matter which mechanism / ε you condition on?

The chart above fixes `mechanism = none` and splits by ε. This cell checks what that choice
costs — how far each generator's accuracy moves when you vary the axis being held fixed. If
a row's range is small, pooling would have been harmless; if it is large, any figure that
pools over that axis is not reporting a single quantity.

In [ ]:
sens = []
for axis in ['epsilon', 'fairness_mechanism']:
    m = u.groupby(['dataset', 'sdg_method', axis]).acc.mean().unstack()
    sens.append(pd.DataFrame({'axis_varied': axis, 'min': m.min(axis=1),
                              'max': m.max(axis=1), 'range': m.max(axis=1) - m.min(axis=1)}))
sens = pd.concat(sens).reset_index().dropna(subset=['range'])
sens = sens.pivot(index=['dataset', 'sdg_method'], columns='axis_varied', values='range')
sens.columns = ['spread across mechanism', 'spread across \u03b5']
display(sens.round(4).style.background_gradient(cmap='Reds'))

print('Survivorship check -- runs that produced a usable classifier, by \u03b5:')
surv = (runs.groupby(['dataset', 'sdg_method', 'eps_label'])
            .agg(attempted=('run_id', 'size'), survived=('acc', 'count')).reset_index())
surv['survival_rate'] = (surv.survived / surv.attempted).round(2)
display(surv[surv.survival_rate < 1])

**Read:** for MST the numbers barely move (spread ≈ 0.001 across ε, 0.007 across mechanism) —
pooling would have been fine. For the causal-GAN variants both spreads reach 0.08–0.10, so
pooling would have blurred away roughly ten accuracy points.

The survivorship table is the more serious issue and the reason the chart is split by ε.
DECAF+DP-GAN's surviving runs are unbalanced across ε in **opposite directions on the two
datasets** (Adult 56/36/24 vs COMPAS 12/24/60 at ε = 1/10/1000). Since its accuracy rises
with ε, a single pooled bar would show COMPAS at its best cell and Adult at its worst — two
different ε mixtures, presented as if comparable. **Any average over a collapsing method is
an average over the runs that happened to work**, which is exactly the direction that
flatters a failing method.

**Read — and this is a sobering table:**

- On **COMPAS**, the marginal methods do real work at *every* privacy level. PrivBayes gains
  +0.092 / +0.099 / +0.108 over the trivial predictor at ε = 1 / 10 / 1000, retaining 96–98%
  of train-on-real accuracy. MST and PrivSyn are close behind. These are genuinely usable
  releases, and the fact that ε=1 barely costs anything is the good news of the whole study.
- On **Adult**, only PrivBayes (+0.028 → +0.066) and PrivSyn (+0.023 → +0.058) clear the
  baseline. **MST is at −0.001, −0.000, −0.001 across all three ε — indistinguishable from
  the trivial predictor at every privacy level.** Its headline 0.747 accuracy looks
  competitive and teaches a model nothing whatsoever about who earns above $50k. This is the
  single most important reason to report lift rather than accuracy.
- The entire causal-GAN family is at or below the baseline on both datasets. DECAF+DP-CTGAN
  on Adult converges *to* the trivial predictor as ε loosens (−0.098 → −0.007 → −0.001),
  which is worth holding onto: it explains why no fairness mechanism moves that
  configuration in Section 7.2. **A model that never learned the outcome has no bias to
  remove.**
- **DECAF+DP-GAN's ε=1 row on COMPAS averages 3 surviving runs out of 15.** Treat it as an
  anecdote, not a measurement.

This does not sink the project — the fairness results below are about *differences between
mechanisms holding the generator fixed*, and those are still valid. But it does mean the
headline framing should be "fairness mechanisms are nearly free" rather than "these
pipelines produce great data", and Adult needs the fix in Section 10.1 before it can carry a
utility claim.

## 7. Fairness — the main event

Everything so far was setup. The question this project exists to answer:

> **If you remove a causal pathway when you generate the data, does a model trained on that
> data come out fairer — and what does it cost?**

### 7.1 The headline: fairness gap by generator × mechanism

Each cell averages over ε, role split and 5 seeds. Collapsed runs are excluded (they have no
classifier, so no gap).

In [ ]:
f = runs[runs.fairness_gap.notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, ds in zip(axes, DATASETS):
    p = (f[f.dataset == ds].pivot_table(index='sdg_method', columns='fairness_mechanism',
                                        values='fairness_gap'))
    p = p.reindex([m for m in METHOD_ORDER if m in p.index])[
        [c for c in MECH_ORDER if c in p.columns]]
    im = ax.imshow(p.values, cmap='RdYlGn_r', aspect='auto', vmin=0,
                   vmax=np.nanmax(p.values))
    ax.set_xticks(range(p.shape[1])); ax.set_xticklabels([nice(c) for c in p.columns])
    ax.set_yticks(range(p.shape[0])); ax.set_yticklabels([nice(i) for i in p.index])
    for i in range(p.shape[0]):
        for j in range(p.shape[1]):
            v = p.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, '%.3f' % v, ha='center', va='center', fontsize=10,
                        color='white' if v > np.nanmax(p.values) * 0.6 else 'black')
    ax.set_title('%s — demographic parity gap (lower = fairer)' % nice(ds))
    ax.set_xlabel('fairness mechanism'); ax.grid(False)
plt.tight_layout(); plt.show()

f.pivot_table(index=['dataset', 'sdg_method'], columns='fairness_mechanism',
              values=['fairness_gap', 'cond_fairness_gap']).round(4)

**Read:** greener is fairer. Two things stand out:

1. **Within almost every row, `none` is the worst column.** The mechanisms do something.
2. **The row-to-row variation is as large as the column-to-column variation.** Which
   generator you choose affects the fairness gap about as much as which mechanism you apply.
   That is worth knowing: fairness is not purely a property of the intervention.

But cell averages hide the thing we actually want, because a lot of the variation here is
seed noise. The next section removes it.

### 7.2 The cost of fairness — paired comparison

This is the most statistically careful result in the notebook and the one to put on a slide.

Instead of comparing averages, we compare **matched pairs**: for each identical
(dataset, generator, ε, role split, seed), take the run with mechanism X and the run with
mechanism `none`, and difference them. Everything else is held exactly constant, so the
seed noise cancels.

- **Δ gap < 0** ⇒ the mechanism made the model fairer. This is the effect we want.
- **Δ accuracy** ⇒ what it cost.
- **`frac_improved`** ⇒ in what fraction of the matched pairs did the gap actually go down.
  A mechanism that helps *on average* but only in 55% of pairs is not something to deploy.

In [ ]:
KEY = ['dataset', 'sdg_method', 'epsilon', 'role_config', 'seed']
METRICS = ['fairness_gap', 'cond_fairness_gap', 'acc', 'tvd_2way']

piv = runs.pivot_table(index=KEY, columns='fairness_mechanism', values=METRICS, dropna=False)

rows = []
for ds in DATASETS:
    for meth in METHOD_ORDER:
        for mech in ['ftu', 'dp', 'cf']:
            try:
                sel = (piv.index.get_level_values('dataset') == ds) & \
                      (piv.index.get_level_values('sdg_method') == meth)
                sub = piv[sel]
                dg = (sub[('fairness_gap', mech)] - sub[('fairness_gap', 'none')]).dropna()
                da = (sub[('acc', mech)] - sub[('acc', 'none')]).dropna()
                dt = (sub[('tvd_2way', mech)] - sub[('tvd_2way', 'none')]).dropna()
            except KeyError:
                continue
            if len(dg) < 3:
                continue
            se = dg.std(ddof=1) / np.sqrt(len(dg))
            # WHERE IT STARTED and WHERE IT ENDED, not just how far it moved.
            # Without these two columns a generator that was already fair looks
            # identical to one the mechanism failed on -- both show ~0 delta.
            b = sub[('fairness_gap', 'none')].dropna()
            base_gap = b.mean()
            rows.append(dict(dataset=ds, method=meth, mechanism=mech, pairs=len(dg),
                             baseline_gap=base_gap,
                             gap_delta=dg.mean(), gap_se=se,
                             final_gap=base_gap + dg.mean(),
                             pct_removed=(-dg.mean() / base_gap * 100) if base_gap > 1e-9 else np.nan,
                             t=dg.mean() / se if se > 0 else np.nan,
                             frac_improved=(dg < 0).mean(),
                             baseline_lift=sub[('acc', 'none')].mean() - MAJORITY[ds],
                             acc_delta=da.mean(), tvd_delta=dt.mean()))
cost = pd.DataFrame(rows)
cost['significant'] = np.where(cost.t.abs() > 2, np.where(cost.gap_delta < 0, 'FAIRER', 'worse'), '-')

# Read left-to-right: started at -> moved by -> ended at -> and was the model
# even worth anything (baseline_lift <= 0 means it never beat guessing).
COLS = ['dataset', 'method', 'mechanism', 'pairs', 'baseline_gap', 'gap_delta',
        'final_gap', 'pct_removed', 'frac_improved', 'acc_delta', 'baseline_lift',
        'significant']
(cost[COLS].round(4).style.hide(axis='index')
     .background_gradient(subset=['gap_delta'], cmap='RdYlGn_r')
     .background_gradient(subset=['final_gap'], cmap='RdYlGn_r'))

In [ ]:
# Effect of each mechanism, with 95% CI -- the slide figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharex=False)
for ax, ds in zip(axes, DATASETS):
    d = cost[cost.dataset == ds].copy()
    d['method'] = pd.Categorical(d.method, METHOD_ORDER, ordered=True)
    d = d.sort_values(['method', 'mechanism'])
    labels, vals, errs, cols = [], [], [], []
    for meth in [m for m in METHOD_ORDER if m in set(d.method)]:
        for mech in ['ftu', 'dp', 'cf']:
            r = d[(d.method == meth) & (d.mechanism == mech)]
            if len(r) == 0:
                continue
            labels.append('%s / %s' % (nice(meth), nice(mech)))
            vals.append(r.gap_delta.iloc[0]); errs.append(1.96 * r.gap_se.iloc[0])
            cols.append(MECH_COLORS[mech])
    y = np.arange(len(labels))
    ax.barh(y, vals, xerr=errs, color=cols, height=0.7, capsize=2.5)
    ax.axvline(0, color='k', lw=1.2)
    ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('\u0394 demographic parity gap vs no mechanism\n\u2190 fairer          worse \u2192')
    ax.set_title('%s — does the mechanism actually help?' % nice(ds))
handles = [plt.Rectangle((0, 0), 1, 1, color=MECH_COLORS[m]) for m in ['ftu', 'dp', 'cf']]
axes[0].legend(handles, ['FTU', 'DP (parity)', 'CF'], fontsize=9, frameon=False, loc='lower left')
fig.suptitle('Bars left of zero = the causal intervention worked (95% CI, matched pairs)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# What did it cost? Fairness gained vs accuracy lost.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ds in zip(axes, DATASETS):
    d = cost[cost.dataset == ds]
    for mech in ['ftu', 'dp', 'cf']:
        s = d[d.mechanism == mech]
        ax.scatter(-s.gap_delta, s.acc_delta, s=110, color=MECH_COLORS[mech],
                   label=nice(mech), edgecolor='k', linewidth=0.5, zorder=3)
        for _, r in s.iterrows():
            ax.annotate(nice(r.method), (-r.gap_delta, r.acc_delta), fontsize=7.5,
                        xytext=(4, 4), textcoords='offset points')
    ax.axhline(0, color='k', lw=1); ax.axvline(0, color='k', lw=1)
    ax.set_xlabel('fairness gained  (gap reduction) \u2192')
    ax.set_ylabel('accuracy change \u2192')
    ax.set_title('%s — the cost of fairness' % nice(ds))
    ax.legend(fontsize=9, frameon=False)
fig.suptitle('Upper-right = free fairness. Most points sit near the zero-cost line.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Average cost of each mechanism, pooled over generators:')
cost.groupby(['dataset', 'mechanism'])[
    ['gap_delta', 'acc_delta', 'tvd_delta', 'frac_improved']].mean().round(4)

In [ ]:
# WHERE EACH GENERATOR STARTS AND WHERE THE MECHANISM LEAVES IT.
# The delta chart above answers 'did the mechanism move it'. This one answers
# 'is the result actually fair', which is a different question with a
# different ranking -- and the one to check before calling a generator bad.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
for ax, ds in zip(axes, DATASETS):
    d = cost[(cost.dataset == ds) & (cost.mechanism == 'dp')].copy()
    d['method'] = pd.Categorical(d.method, METHOD_ORDER, ordered=True)
    d = d.sort_values('method')
    y = np.arange(len(d))
    for yi, (_, r) in zip(y, d.iterrows()):
        ax.plot([r.baseline_gap, r.final_gap], [yi, yi], color='#999999', lw=2.2,
                zorder=1, solid_capstyle='round')
        ax.scatter(r.baseline_gap, yi, s=95, color='#BBBBBB', edgecolor='k',
                   linewidth=0.6, zorder=3)
        ax.scatter(r.final_gap, yi, s=95, color=COLORS[r.method], edgecolor='k',
                   linewidth=0.6, zorder=3)
        # Flag the generators whose 'fairness' is really just failure to learn.
        if r.baseline_lift <= 0:
            ax.annotate('never beat guessing', (max(r.baseline_gap, r.final_gap), yi),
                        xytext=(8, -3), textcoords='offset points', fontsize=7.5,
                        color='#B22222', style='italic')
    ax.set_yticks(y); ax.set_yticklabels([nice(m) for m in d.method], fontsize=9.5)
    ax.invert_yaxis()
    ax.axvline(0.05, color='#B22222', ls=':', lw=1.2)
    ax.annotate('0.05', xy=(0.05, 0), xycoords=('data', 'axes fraction'),
                xytext=(3, 4), textcoords='offset points', color='#B22222',
                fontsize=8, va='bottom')
    ax.set_xlim(left=min(0, ax.get_xlim()[0]))
    ax.set_xlabel('demographic parity gap  ← fairer')
    ax.set_title('%s — start (grey) → after DP mechanism (colour)' % nice(ds))
fig.suptitle('A short arrow can mean “already fair”, not “mechanism failed”',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Ranked by WHERE THEY END UP (DP mechanism), not by how far they moved:')
display(cost[cost.mechanism == 'dp']
        .sort_values(['dataset', 'final_gap'])
        [['dataset', 'method', 'baseline_gap', 'gap_delta', 'final_gap',
          'pct_removed', 'baseline_lift']].round(4)
        .style.hide(axis='index'))

**Read — this is the core claim, and the caveats that come with it:**

**The DP (demographic parity) mechanism works, reliably and almost for free.** Pooled over
generators it cuts the gap by **−0.091 on Adult** and **−0.142 on COMPAS**, improving the gap
in 68% / 70% of matched pairs. The accuracy cost is **−0.016 on Adult** and
**−0.048 on COMPAS**. The strongest individual effects (COMPAS DECAF −0.325, COMPAS DP-GAN
−0.314, Adult DECAF+CTGAN −0.187) are many standard errors from zero. Note the accuracycost is no longer "literally zero" — the corrected epoch settings gave these generators real
predictive signal, and taking the bias out of a model that actually predicts something costs
more than taking it out of one that does not.

**CF sits where the theory says it should** — between `FTU` and `DP` (−0.074 Adult, −0.115
COMPAS), because it deliberately leaves the disparity that flows through admissible
attributes intact. That the ordering `none` > FTU > CF > DP emerges from the data rather than
being imposed is a good sign for the framework.

**FTU is weak and unreliable, exactly as the fairness literature predicts.** Pooled over
Adult it improves the gap in only **35% of matched pairs** — it makes things *worse* more
often than better. Removing the direct edge just reroutes the influence through proxies.
This is a useful confirmatory result: the cheap intervention does not work, and the causal
one does.

**About MST's small deltas — the reframe, and why it does not end where you would expect.**

MST posts the smallest Δ of any generator on Adult (DP mechanism: **−0.018**, against DECAF's
−0.180). Presented as a delta alone that reads as "the mechanism failed on MST", and that
reading is wrong: MST's Adult baseline gap is only **0.021**, so 0.018 is **88% of all the
gap there was to remove**, and it *ends* at **0.0024 — among the lowest final gaps in the
table** (Adult DECAF ends lower still, at 0.0007, but see the lift caveat below — it is
0.028 *below* the majority baseline, so its perfect-looking gap is worth nothing).
A generator cannot remove gap it never had. This is the trap from Section 2.4b, and it is why
the table and the second chart above now report start, move, and end together.

**But the honest version does not stop there.** Check the `baseline_lift` column: MST on
Adult scores **−0.0007** — it never beats simply guessing the majority class. So its tiny gap
is not the reward for being well-behaved; it is the *trivially-fair failure mode* from
Section 2.4 — a model with no signal has no disparity to show. MST is not the fairness winner
on Adult. It is a near-constant predictor that is fair for the uninteresting reason.

The test is COMPAS, where MST genuinely learns (**lift +0.082**, second only to PrivBayes at
+0.100).
There its baseline gap is **0.233** and the DP mechanism removes only **18%** of it, ending at
**0.190** — 4th of 7 methods. So on the dataset where MST has real predictive signal, the
small delta *is* a real weakness, not an artefact of an already-fair starting point.

**The general lesson, which applies to every row:** a fairness number is only interpretable
alongside (a) the level it started from and (b) whether the model learned anything at all.
Two of the seven generators here have a near-zero gap purely because they are near-constant
predictors. Both charts above now mark those explicitly.

**Two null results that must be reported honestly:**

- **FTU on MST/Adult is an exact no-op** — Δ = 0.0000 with *zero* standard error across all
  45 matched pairs. Not noise: MST's private structure search never selects the direct
  sex→income marginal in the first place, so there is no edge for FTU to remove.
  Fairness-through-unawareness is vacuous whenever the generator was never going to use the
  direct edge anyway.
- **No mechanism does anything on DECAF+DP-CTGAN/Adult** (Δ ≈ +0.005, all |t| < 1.1). The
  reason is visible one section up: that configuration's baseline gap is already 0.007,
  because its accuracy is *below* the majority baseline. **There is no bias to remove from a
  model that has not learned anything.** This is the "trivially fair" failure mode from
  Section 2.4 caught in the act, and it is why fairness numbers are never quoted here without
  the accompanying lift.

### 7.3 The Distributional Fairness axis — does the synthetic data tell the truth about itself?

From Section 2.5: score the same trained classifier against the **real** holdout (ground
truth) and against the **synthetic** data (what an analyst without data access would see).
The difference is the **audit error**.

#### How to read the scatter plot below

**Every dot is one experiment run** — one (dataset, generator, mechanism, ε, role split, seed)
combination. Each run gives us the *same model's* fairness gap measured two different ways,
and those two numbers are the dot's two coordinates:

- **x = the gap measured on real held-out people.** The truth. How the model will behave when
  deployed.
- **y = the gap measured on the synthetic data itself.** What someone auditing the public
  release for bias — with no access to the real data — would report.

**The dashed diagonal is the line y = x**, i.e. *the two measurements agree*. It is not a fit,
a trend, or a regression through the points — it is drawn at 45° regardless of the data, as a
reference for perfection. A generator whose release is a trustworthy audit substrate puts its
dots **on** that line.

So the vertical distance from a dot to the diagonal *is* the audit error, and the side tells
you the direction of the lie:

| Where the dot sits | What it means | How bad |
|---|---|---|
| **On** the diagonal | auditing the synthetic release gives the right answer | ideal |
| **Below** the diagonal (y < x) | the release looks fairer than the model really is | **dangerous** — ship a biased model believing it is clean |
| **Above** the diagonal (y > x) | the release looks less fair than reality | wasteful, but fails safe |

In [ ]:
df_ax = runs[runs.max_abs_dp_gap__real.notna() & runs.max_abs_dp_gap__synth.notna()].copy()
df_ax['audit_error'] = (df_ax.max_abs_dp_gap__synth - df_ax.max_abs_dp_gap__real)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
for ax, ds in zip(axes, DATASETS):
    d = df_ax[df_ax.dataset == ds]
    for m, sub in d.groupby('sdg_method'):
        ax.scatter(sub.max_abs_dp_gap__real, sub.max_abs_dp_gap__synth, s=16, alpha=0.45,
                   color=COLORS[m], label=nice(m))
    lim = max(d.max_abs_dp_gap__real.max(), d.max_abs_dp_gap__synth.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'k--', lw=1.4)
    ax.text(lim * 0.55, lim * 0.60, 'perfect agreement', rotation=38, fontsize=9)
    ax.set_xlabel('gap measured on REAL data  (ground truth)')
    ax.set_ylabel('gap measured on SYNTHETIC data  (what an auditor sees)')
    ax.set_title('%s — can you audit fairness from the release alone?' % nice(ds))
    ax.legend(fontsize=8, frameon=False, markerscale=1.6)
fig.suptitle('Points below the line = the synthetic data UNDER-reports its own unfairness',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

audit = (df_ax.groupby(['dataset', 'sdg_method'])
          .agg(real_gap=('max_abs_dp_gap__real', 'mean'),
               synth_gap=('max_abs_dp_gap__synth', 'mean'),
               mean_audit_error=('audit_error', 'mean'),
               mean_abs_error=('audit_error', lambda s: s.abs().mean()),
               pct_underreport=('audit_error', lambda s: (s < 0).mean()))
          .round(4).reset_index())
audit.style.hide(axis='index').background_gradient(subset=['mean_abs_error'], cmap='Reds')

**Read — the errors are large and they point the wrong way.**

Where `mean_audit_error` is negative, the synthetic data makes the model look **fairer than
it really is**. That is the dangerous direction: an organisation audits on the release, sees
an acceptable gap, and ships a model that is more biased in deployment. Most generators land
there:

| Generator | Real gap | Gap an auditor would measure | Under-reports in |
|---|---|---|---|
| PrivBayes (COMPAS) | 0.246 | 0.190 | **81%** of runs |
| PrivSyn (Adult) | 0.160 | 0.100 | **79%** of runs |
| DECAF+CTGAN (Adult) | 0.123 | 0.109 | 58% of runs |
| PrivBayes (Adult) | 0.141 | 0.088 | 72% of runs |
| **MST (Adult)** | **0.015** | **0.018** | mean abs. error **0.004** |

PrivSyn on Adult reports a gap of 0.100 when the truth is 0.160 — it hides **38% of the
actual disparity**. An auditor using that release would systematically under-count bias.

**MST is the standout in the other direction:** its audit error is 0.004, an order of
magnitude smaller than anyone else's. MST is simultaneously the most faithful generator
(Section 4.1) and the most *honest about itself*. That is not a coincidence worth assuming —
but it is a strong hypothesis: fidelity to the joint distribution is what makes a release
self-diagnosing.

This axis is under-exploited in the literature and is a strong candidate for the paper's
second contribution: **"a synthetic release is not a valid substrate for fairness auditing
unless you measure this, and most releases fail."**

### 7.4 How much of this is just noise? The seed floor

**What a "cell" is.** The experiment is a grid, and one **cell** is one fully-specified
combination of knobs: *(dataset, generator, ε, fairness mechanism, role split)* — for example
*(Adult, MST, ε=10, CF, prefair split)*. That is a single experimental condition.

**What a "seed" is.** We ran each cell **5 times** with 5 different random seeds. The seed
changes the train/holdout split, the generator's random initialisation and sampling, and the
downstream classifier's initialisation — but **nothing about the experimental condition**. So
any difference between the 5 runs of one cell is *pure chance*: the same experiment, run
again, landing somewhere slightly different.

**What "seed noise" therefore is.** The spread of those 5 results. It is the measurement
error of this entire pipeline — the amount a number can move for no reason at all.

**Why you need it before believing anything.** If re-running the identical condition swings
the fairness gap by ±0.05, then a mechanism that shrinks the gap by 0.03 has produced an
effect *smaller than the noise*, and a single run showing it would be meaningless. The
histogram below measures that floor directly: for every cell, the standard deviation of the
fairness gap across its 5 seeds.

In [ ]:
cell = ['dataset', 'sdg_method', 'epsilon', 'fairness_mechanism', 'role_config']
sd = runs.groupby(cell, dropna=False).fairness_gap.agg(['mean', 'std', 'size']).dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for ax, ds in zip(axes, DATASETS):
    s = sd[sd.index.get_level_values('dataset') == ds]['std']
    ax.hist(s, bins=22, color='#4C72B0', alpha=0.85)
    ax.axvline(s.median(), color='crimson', lw=2)
    ax.text(s.median() * 1.06, ax.get_ylim()[1] * 0.85,
            'median = %.3f' % s.median(), color='crimson', fontsize=10)
    ax.set_xlabel('within-cell std. dev. of the fairness gap across 5 seeds')
    ax.set_ylabel('number of grid cells')
    ax.set_title('%s — seed noise floor' % nice(ds))
fig.suptitle('Any effect smaller than this needs matched pairs to detect',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

print('Within-cell std of fairness_gap, by dataset:')
display(sd.reset_index().groupby('dataset')['std'].describe().round(4))
print('\nThis is exactly why Section 7.2 uses matched pairs: pooling 45 pairs drops the')
print('standard error to ~0.02, which is small enough to resolve the mechanism effects.')

**Read:** the median cell-level standard deviation is substantial — on COMPAS it is roughly
the size of the mechanism effects themselves. **A single-seed experiment on this grid would
be uninterpretable.** This alone justifies the 5-seed design and is worth a sentence in the
paper's experimental setup.

### 7.5 Does the answer depend on who you call protected?

The prediction from Section 3: CF blocks only the pathways that *don't* route through an
admissible attribute, so widening the admissible set should make CF do **less** work and
leave a **larger** residual gap.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ds in zip(axes, DATASETS):
    d = f[f.dataset == ds]
    p = d.pivot_table(index='role_config', columns='fairness_mechanism', values='fairness_gap')
    p = p[[c for c in MECH_ORDER if c in p.columns]]
    x = np.arange(len(p.index)); w = 0.2
    for i, mech in enumerate(p.columns):
        ax.bar(x + (i - 1.5) * w, p[mech], w, label=nice(mech), color=MECH_COLORS[mech])
    ax.set_xticks(x); ax.set_xticklabels(p.index, rotation=12, ha='right', fontsize=9)
    ax.set_ylabel('demographic parity gap (lower = fairer)')
    ax.set_title('%s — sensitivity to the role split' % nice(ds))
    ax.legend(fontsize=9, frameon=False, title='mechanism')
fig.suptitle('The conclusion is robust to who you protect; the magnitude is not',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

f.pivot_table(index=['dataset', 'role_config'], columns='fairness_mechanism',
              values=['fairness_gap', 'cond_fairness_gap', 'acc']).round(4)

**Read:** the *ordering* of the mechanisms is stable across role splits — that is the
reassuring part, and it means the qualitative claim does not hinge on a modelling choice.
The *magnitudes* move a lot, which is the honest caveat: absolute gap numbers are only
meaningful relative to a stated role split, and any headline number must name its split.

### 7.6 Parity vs. error-rate fairness — you cannot have both

#### The two definitions, in one sentence each

There is no single definition of "fair". These are the two that matter most, and they can
disagree about the same model:

- **Parity fairness** (demographic parity, the x-axis) — *does the model say yes to both
  groups at the same rate?* It ignores whether the answers are correct. Its complaint is
  **"you approve fewer women."**
- **Error-rate fairness** (TPRB / equalised odds, the y-axis) — *is the model equally
  accurate for both groups?* It ignores the overall approval rates. Its complaint is
  **"you miss more of the qualified women than the qualified men."**

A concrete case where they disagree: hand out approvals to exactly 30% of each group, chosen
at random within groups. Parity is now perfect. But error rates are dreadful and unequal,
because you approved qualified and unqualified people indiscriminately. Perfect on one
definition, terrible on the other — from a single model.

**Why they are said to be incompatible.** When the groups have genuinely different base rates
(in Adult, more men than women really do earn over \$50k), it is a *theorem* that you cannot
have both exactly, short of a perfect classifier. Section 2.4 sketches why.

#### How to read the scatter below

**One dot = one experiment run**, coloured by which fairness mechanism it used. Its position
is that single run scored under *both* definitions at once:

- **x = parity gap** (lower = the two groups get approved at similar rates)
- **y = TPRB** (lower = the model is similarly accurate for both groups)

**Bottom-left is good on both counts.** The interesting question is the *shape* of the cloud:
if the two definitions really traded off, the dots would form a **downward** slope — pushing
x down would push y up. If they move together, the cloud slopes **upward**, and the printed
correlation `r` is the number that settles which. Note the correlation here describes *how
our configurations happen to be distributed*; it is not a claim about the theorem.

In [ ]:
e = runs[runs.fairness_gap.notna() & runs.max_abs_tprb__real.notna()]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ds in zip(axes, DATASETS):
    d = e[e.dataset == ds]
    for mech in MECH_ORDER:
        s = d[d.fairness_mechanism == mech]
        ax.scatter(s.fairness_gap, s.max_abs_tprb__real, s=18, alpha=0.5,
                   color=MECH_COLORS[mech], label=nice(mech))
    ax.set_xlabel('demographic parity gap (equal YES rates)')
    ax.set_ylabel('TPRB (equal catch rates among true positives)')
    ax.set_title('%s — two definitions of fair' % nice(ds))
    ax.legend(fontsize=9, frameon=False, title='mechanism', markerscale=1.5)
fig.suptitle('In our runs the cloud slopes UP: the two move together, they do not trade off',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print('Correlation between the two fairness notions (per dataset):')
for ds in DATASETS:
    d = e[e.dataset == ds]
    print('  %-7s  r = %+.3f' % (ds, d.fairness_gap.corr(d.max_abs_tprb__real)))

e.groupby(['dataset', 'fairness_mechanism'])[
    ['fairness_gap', 'cond_fairness_gap', 'max_abs_tprb__real',
     'max_abs_tnrb__real']].mean().round(4)

**Read — and this came out the opposite of what the theory would lead you to expect.** The
two fairness notions are almost perfectly correlated in our runs: **r = +0.95 on Adult and
+0.97 on COMPAS**. A configuration that improves demographic parity improves TPRB too,
essentially one-for-one.

The impossibility result is not violated — it says you cannot satisfy both *exactly* when
base rates differ, and none of our configurations satisfy either exactly. What we are seeing
is that in the regime we actually operate in (gaps of 0.05–0.3, not 0), the causal
intervention shrinks disparity of *all* kinds at once, because it is removing the underlying
pathway rather than post-hoc rebalancing a decision threshold.

**That is an argument for the causal approach, and it is worth making explicitly in the
paper:** threshold-based fairness repair forces you to pick which fairness definition to
satisfy, because it trades one against the other by construction. Editing the generative
process appears not to. It is a testable claim, and the head-to-head against post-hoc
fairness repair proposed in Section 10.3 is the experiment that would settle it.

### 7.7 Measuring fairness the way the DECAF paper defines it (their Definition 4)

**Short answer to "can we measure DECAF's Definition 4?": we already do — it is the axis
running through Sections 2.5 and 7.3, and every run in the grid carries it.** This section
makes the correspondence explicit and then uses it to ask a question the earlier sections
did not.

#### What Definition 4 actually says

DECAF's first three definitions are about *the graph*: FTU (Def. 1) removes the direct
protected→outcome edge, demographic parity (Def. 2) removes every such path, and conditional
fairness (Def. 3) removes the paths not routed through an admissible attribute. Those are the
three mechanisms in this project's `fairness_mechanism` column.

**Definition 4 — *distributional* fairness — is a different kind of statement.** It is not a
fourth way to edit the graph; it is a statement about **which population you check fairness
against** once a downstream model has been trained on the synthetic data $P'(X)$. There are
two choices, and DECAF is emphatic that they are not equally interesting:

| | Check the model against… | DECAF's view | Our column |
|---|---|---|---|
| **DF-ORIGINAL** | the **real** distribution $P(X)$ | the case that matters — the model has to stay fair when it meets the real world | `max_abs_dp_gap__real` |
| **DF-SYNTHETIC** | the synthetic distribution $P'(X)$ | *"uninteresting"* (their §4.2): trivially satisfiable, e.g. by randomising the protected column, and guarantees nothing about deployment | `max_abs_dp_gap__synth` |

Because it is orthogonal to FTU/DP/CF — any mechanism can be paired with either reference —
this project logs it as **its own axis** (`eval_reference`) rather than as a fourth mechanism,
and computes **both references from a single generation** so they are exactly comparable.

#### The question that gets us

If DF-SYNTHETIC is trivially satisfiable and DF-ORIGINAL is the one that matters, then the
failure mode to worry about is a release that **passes the easy test and fails the real one**:
the synthetic data says the model is fair, the real world says it is not. Call it a **false
assurance**. Below: how often does each mechanism produce one?

In [ ]:
# DECAF Def. 4, both references, from the same generation.
# A run 'passes' a reference if its parity gap there is at most FAIR_THRESHOLD.
FAIR_THRESHOLD = 0.05

d4 = runs[runs.max_abs_dp_gap__real.notna() & runs.max_abs_dp_gap__synth.notna()].copy()
d4['pass_synthetic'] = d4.max_abs_dp_gap__synth <= FAIR_THRESHOLD   # the easy test
d4['pass_original'] = d4.max_abs_dp_gap__real <= FAIR_THRESHOLD     # the one that matters
d4['false_assurance'] = d4.pass_synthetic & ~d4.pass_original

tab = (d4.groupby(['dataset', 'fairness_mechanism'])
        .agg(runs=('run_id', 'size'),
             gap_synthetic=('max_abs_dp_gap__synth', 'mean'),
             gap_original=('max_abs_dp_gap__real', 'mean'),
             pass_synthetic=('pass_synthetic', 'mean'),
             pass_original=('pass_original', 'mean'),
             false_assurance=('false_assurance', 'mean'))
        .round(4).reset_index())
tab['mechanism'] = pd.Categorical(tab.fairness_mechanism, MECH_ORDER, ordered=True)
tab = tab.sort_values(['dataset', 'mechanism']).drop(columns='mechanism')
display(tab.style.hide(axis='index')
        .background_gradient(subset=['false_assurance'], cmap='Reds'))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
for ax, ds in zip(axes, DATASETS):
    t = tab[tab.dataset == ds].set_index('fairness_mechanism').reindex(MECH_ORDER)
    x = np.arange(len(MECH_ORDER)); w = 0.38
    ax.bar(x - w/2, t.gap_synthetic, w, color='#8FBBD9', edgecolor='k', linewidth=0.5,
           label='DF-SYNTHETIC  (the "easy" reference)')
    ax.bar(x + w/2, t.gap_original, w, color='#C44E52', edgecolor='k', linewidth=0.5,
           label='DF-ORIGINAL  (the one that matters)')
    for xi, (a, b) in enumerate(zip(t.gap_synthetic, t.gap_original)):
        if pd.notna(a) and pd.notna(b) and b > 0:
            ax.annotate('%.0f%% understated' % ((b - a) / b * 100),
                        (xi, max(a, b)), xytext=(0, 6), textcoords='offset points',
                        ha='center', fontsize=8.5, color='#B22222')
    ax.axhline(FAIR_THRESHOLD, color='k', ls=':', lw=1.2)
    ax.set_xticks(x); ax.set_xticklabels([nice(m) for m in MECH_ORDER])
    ax.set_ylabel('mean demographic parity gap')
    ax.set_xlabel('fairness mechanism')
    ax.set_title('%s — the gap depends on which population you check' % nice(ds))
    ax.legend(fontsize=8.5, frameon=False)
fig.suptitle('DECAF Def. 4: the mechanism that works best is also the one that '
             'most overstates its own success', fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Read — this is the sharpest result in the notebook, and it is not the one we expected.**

**Every mechanism looks better on the easy reference than on the real one.** The bars never
tie. Checking a released dataset against itself always flatters it.

**The effect is largest for the mechanism that works best.** The DP mechanism drives the
synthetic-reference gap down to **0.013 on Adult**, which looks like near-perfect fairness.
Measured against real people the same models sit at **0.044** — the true gap is **3.4× the
reported one**. On COMPAS it is 0.112 reported against 0.172 real.

**And so the false-assurance rate is *highest* for DP, not lowest:**

| Mechanism | Adult | COMPAS |
|---|---|---|
| none | 5.6% | 2.2% |
| FTU | 5.6% | 3.5% |
| CF | 14.3% | 9.6% |
| **DP** | **17.7%** | **18.9%** |

Roughly **one run in five** using the best-performing fairness mechanism produces a release
that certifies itself as fair while the underlying model is not. Applying no mechanism at all
is *less* misleading — because it never looked fair in the first place.

**Why this happens, and why it is not an argument against the mechanisms.** The mechanisms
act on the generative process, so the synthetic data is exactly where their effect is
strongest and most completely realised. The downstream model then meets real data, which
still contains every proxy pathway the generator suppressed, and part of the disparity comes
back. The mechanism genuinely helped — Section 7.2 measures that against the real reference
and the improvement is real. What fails is the **self-report**.

**The claim this supports:** *a fairness intervention at generation time must be validated
against the real distribution; validating it on the synthetic release systematically
overstates it, and overstates it most exactly when the intervention is working.* DECAF states
the ORIGINAL/SYNTHETIC distinction and then evaluates the ORIGINAL case; to our knowledge
nobody has measured the size of the discrepancy across generators and mechanisms. That is a
contribution, and this notebook already has the data for it.

*(Caveat, since it cuts the other way too: `decaf_dpctgan` scores near-zero on both
references not because it is fair but because it never learned the outcome. Its rows are
excluded from nothing here, so read this table with Section 7.2's `baseline_lift` column
in hand.)*

### 7.8 Does any of this survive a bigger causal graph? First SNAKE and SBO results

Section 1.1 posed the question this project most needs answered: the fairness mechanisms all
work by **cutting edges**, so does their effect hold up when there is much more graph to cut?
Adult and COMPAS cannot tell us — 15 and 37 protected→outcome paths is not much of a range.

**These results just landed** (batch `newdata-2026-08-03`, 216 runs, MST/PrivBayes/PrivSyn on
SNAKE and SBO). Read them as a first look, not a settled finding — the caveats are at the
bottom and they are not small.

In [ ]:
nd = pd.read_csv(DATA / 'newdata_runs.csv')
nb_ = pd.read_csv(DATA / 'newdata_baselines.csv')
NEW_MAJ = nb_.groupby('dataset').majority_baseline.mean().to_dict()
NEW_TRTR = nb_.groupby('dataset').trtr_mlp.mean().to_dict()

# Only the three marginal methods ran on the new datasets, so the comparison
# against Adult/COMPAS is restricted to those -- otherwise 'bigger graph'
# would be confounded with 'different set of generators'.
MARG = ['mst', 'privbayes', 'privsyn']
PATHS = {'compas': 15, 'adult': 37, 'snake': 76, 'sbo': 465}

print('Utility on the new datasets (mechanism = none, averaged over eps and role split):')
u = nd[nd.fairness_mechanism == 'none'].groupby(['dataset', 'sdg_method']).agg(
        acc=('downstream_accuracy_mlp', 'mean'), tvd2=('tvd_2way', 'mean')).reset_index()
u['majority'] = u.dataset.map(NEW_MAJ)
u['train_on_real'] = u.dataset.map(NEW_TRTR)
u['lift'] = u.acc - u.majority
u['pct_of_real'] = u.acc / u.train_on_real * 100
display(u.round(4).style.hide(axis='index')
        .background_gradient(subset=['lift'], cmap='RdYlGn'))

rows = []
for src in (runs, nd):
    d = src[src.sdg_method.isin(MARG)]
    KEY = ['dataset', 'sdg_method', 'epsilon', 'role_config', 'seed']
    piv = d.pivot_table(index=KEY, columns='fairness_mechanism',
                        values='fairness_gap', dropna=False)
    for ds, sub in piv.groupby(level='dataset'):
        for mech in ['ftu', 'dp', 'cf']:
            if mech not in sub or 'none' not in sub:
                continue
            dg = (sub[mech] - sub['none']).dropna()
            if len(dg) < 3:
                continue
            base = sub['none'].mean()
            rows.append(dict(dataset=ds, paths=PATHS[ds], mechanism=mech,
                             pairs=len(dg), baseline=base, delta=dg.mean(),
                             pct_removed=-dg.mean() / base * 100,
                             frac_improved=(dg < -1e-9).mean()))
scale = pd.DataFrame(rows).sort_values(['mechanism', 'paths'])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5))
for mech in ['ftu', 'dp', 'cf']:
    m = scale[scale.mechanism == mech].sort_values('paths')
    axes[0].plot(m.paths, m.pct_removed, 'o-', color=MECH_COLORS[mech],
                 label=nice(mech), lw=2, ms=8)
    axes[1].plot(m.paths, m.frac_improved, 'o-', color=MECH_COLORS[mech],
                 label=nice(mech), lw=2, ms=8)
for ax, ylab, ttl in [(axes[0], '% of the gap removed', 'How much does the mechanism remove?'),
                      (axes[1], 'fraction of runs where the gap went DOWN',
                       'How reliably does it help?')]:
    ax.set_xscale('log')
    ax.set_xticks(list(PATHS.values()))
    ax.set_xticklabels(['%s\n(%d)' % (nice(k), v) for k, v in PATHS.items()], fontsize=8.5)
    ax.set_xlabel('dataset, by number of protected\u2192outcome causal paths')
    ax.set_ylabel(ylab); ax.set_title(ttl, fontsize=11.5)
    ax.legend(fontsize=9, frameon=False)
axes[1].axhline(0.5, color='crimson', ls='--', lw=1.3)
axes[1].text(20, 0.51, 'coin flip', color='crimson', fontsize=8.5)
fig.suptitle('Marginal methods only. SNAKE/SBO are 1 seed \u2014 preliminary.',
             fontsize=12.5, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

display(scale.round(4).style.hide(axis='index')
        .background_gradient(subset=['pct_removed'], cmap='RdYlGn'))

**Read — one clear win, one warning, and a caveat that limits both.**

**The win: SBO is the best-behaved dataset in the whole project on data quality.** MST reaches
2-way TVD **0.036** and downstream accuracy **0.870** against a train-on-real ceiling of
**0.878** — that is **99% of the real data's predictive value retained**, at ε=1, on a
25-attribute table. Compare Adult, where several methods cannot beat the trivial predictor at
all. Wider, lower-cardinality tables are evidently *easier* for the marginal synthesizers, not
harder, which is worth knowing before anyone assumes scale is the problem.

**The warning: the mechanisms do not scale with the graph the way we hoped.** The DP mechanism
removes 31% / 51% / **78%** / **25%** of the gap on COMPAS / Adult / SNAKE / SBO. It is not a
clean decline — SNAKE, the second-largest graph, is where it works *best* — but SBO, with
**465 paths**, is comfortably the worst of the four despite having the second-largest baseline
gap to work with.

**CF's reliability is the sharper signal.** The fraction of runs in which CF actually reduced
the gap falls **0.61 → 0.68 → 0.48 → 0.41** across the four datasets. On SBO, applying CF is
*worse than a coin flip* at making the model fairer. On MST/SBO specifically, CF and FTU are
**exact no-ops** — the gap is identical to `none` to four decimals — which is the same vacuity
already documented for FTU on MST/Adult in Section 7.2: MST's private structure search never
selected the edges those mechanisms would have cut, so there was nothing to remove.

**A plausible mechanism for the SBO result**, worth testing rather than asserting: with 465
paths and 5 protected attributes, blocking the protected→outcome routes still leaves the
outcome highly predictable from `SECTOR`, `EMPLOYMENT` and `PAYROLL`, which are themselves
heavily stratified by owner demographics. The edges get cut; the information arrives anyway.
If that is right, it is an argument that **graph-surgery fairness has a scale limit**, and
finding that limit is a paper in itself.

> #### ⚠️ Why this is preliminary, stated plainly
> - **SNAKE and SBO are 1 seed; Adult and COMPAS are 5.** Section 7.4 measured the seed noise
>   floor at roughly the size of these effects, so the two new points carry error bars we have
>   not yet drawn. This is the single biggest reason not to quote these numbers yet.
> - **Four datasets is not a trend.** "Number of causal paths" is confounded with dataset,
>   role split, base rate, and attribute cardinality. The clean version of this experiment
>   holds the dataset fixed and *prunes its own graph* to vary path count — which is now the
>   top item in Section 10.
> - **Only the three marginal methods ran.** No DECAF variant has touched SNAKE or SBO, so
>   nothing here speaks to the causal-GAN family.
> - The SBO causal DAG is **our construction**, like Adult's, and 68 asserted edges is a lot
>   of modelling assumption to rest a conclusion on.

## 8. Making causal GANs private — the methods contribution

DECAF is the only generator in this study that implements causal fairness *natively*: its
generator is factored along a DAG, so removing an edge is a structural operation rather than
a post-hoc correction. That makes it the natural vehicle for this project.

**The problem: DECAF has no privacy mechanism at all.** It is a WGAN-GP. If causal fairness
is only available from a generator that cannot offer differential privacy, the whole
fairness-privacy story falls apart.

So we built three backbones that keep DECAF's causal generator and swap what is underneath:

| Variant | Representation | Privacy | Purpose |
|---|---|---|---|
| **DECAF+CTGAN** | CTGAN-style: one-hot column blocks, gumbel-softmax heads, conditional vectors with training-by-sampling, PacGAN critic | none | Does a better *representation* alone fix DECAF's fidelity? |
| **DECAF+DP-GAN** | DECAF's original | DP-SGD on the critic (clip + Gaussian noise), RDP accountant | The straightforward way to make it private |
| **DECAF+DP-CTGAN** | CTGAN-style | DP-SGD on the critic | Both at once |

**Honest deviations, all forced by the privacy requirement** (these belong in the paper):

- WGAN-GP's gradient penalty is not DP-compatible (it differentiates through a mixed batch),
  so the DP variants use **weight clipping at 0.01** instead. This is the main source of the
  instability seen below.
- **PacGAN is disabled** (`pac=1`) in DP-CTGAN: packing several records into one critic
  decision breaks the per-record sensitivity analysis.
- **Conditional vectors are disabled** in DP-CTGAN: the log-frequency category weights and
  the "draw a real row matching this condition" step both read private counts on an
  unaccounted path.

Generator updates are free under DP — they are post-processing of an already-private critic.

### 8.1 The result: DP-SGD destroys the causal generator, and the representation rescues it

In [ ]:
gan = runs[runs.sdg_method.str.startswith('decaf')]
col = (gan.groupby(['dataset', 'sdg_method', 'eps_label'])
          .agg(runs=('run_id', 'size'), collapsed=('collapsed', 'sum')).reset_index())
col['collapse_rate'] = col.collapsed / col.runs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ds in zip(axes, DATASETS):
    d = col[col.dataset == ds].copy()
    d['sdg_method'] = pd.Categorical(d.sdg_method, METHOD_ORDER, ordered=True)
    d['eps_label'] = pd.Categorical(d.eps_label, EPS_ORDER, ordered=True)
    d = d.sort_values(['sdg_method', 'eps_label'])
    labels = ['%s\n\u03b5=%s' % (nice(r.sdg_method), r.eps_label) for _, r in d.iterrows()]
    x = np.arange(len(d))
    ax.bar(x, d.collapse_rate, color=['#C44E52' if v > 0 else '#55A868'
                                      for v in d.collapse_rate], width=0.68)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5, rotation=45, ha='right')
    ax.set_ylabel('fraction of runs that collapsed to a single class')
    ax.set_ylim(0, 1.05)
    ax.set_title('%s — generator collapse rate' % nice(ds))
    for xi, v in zip(x, d.collapse_rate):
        ax.text(xi, v + 0.02, '%.0f%%' % (v * 100), ha='center', fontsize=9)
fig.suptitle('DP-GAN collapses constantly. DP-CTGAN never collapses. Same privacy budget.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

summary = (gan.groupby('sdg_method')
              .agg(total_runs=('run_id', 'size'), collapsed=('collapsed', 'sum')).reset_index())
summary['collapse_rate'] = (summary.collapsed / summary.total_runs).round(3)
display(summary)
col.round(3)

**This is the strongest result in the project, and the re-run strengthened it.**

- **DECAF+DP-GAN collapsed in 168 of 360 runs (47%).** Under DP-SGD, the weight-clipped
  critic gives the causal generator a signal too weak to preserve the outcome variable, and
  it degenerates to emitting a single class. A single-class table is worthless: no model can
  be trained, no fairness can be measured.
- **DECAF+DP-CTGAN collapsed in 0 of 360 runs**, at *identical* privacy budgets — including
  ε=1, and including on Adult where DP-GAN fails hardest. This held across both batches
  and both epoch settings: 720 private runs now, zero collapses.

The only difference between them is the data representation and the loss on the generator's
output heads. **The claim: private causal generative modelling of tabular data is not blocked
by DP-SGD, it is blocked by the representation.** That is a clean, falsifiable, novel
statement, and it is the kind of thing a methods paper is built on.

**The ε pattern changed under the corrected epochs, and is worth reporting carefully.**
The old batch showed an *inversion* on Adult — collapse getting worse as ε loosened. That
was an artefact of the wrong epoch count and it is gone. What replaces it is two different
regimes:

| | ε=1 | ε=10 | ε=1000 |
|---|---|---|---|
| **COMPAS** DP-GAN | **100%** | 60% | **0%** |
| **Adult** DP-GAN | 40% | 40% | 40% |

On COMPAS collapse is now cleanly **monotone in the privacy budget** — every single run dies
at ε=1, none at ε=1000. That is the textbook dose-response you would predict, and the
old batch did not show it. On Adult collapse is **completely flat at 40%**, independent of
ε. Taken together: the weight-clipped critic is unstable *on its own*, and DP noise adds
a second, dataset-dependent failure on top of it. Neither dataset alone would have told us
that.

### 8.2 What the survivors actually produce

In [ ]:
g = gan[gan.status == 'done']
t = (g.groupby(['dataset', 'sdg_method', 'eps_label'])
      .agg(tvd_2way=('tvd_2way', 'mean'), acc=('acc', 'mean'), lift=('lift', 'mean'),
           fairness_gap=('fairness_gap', 'mean'), n=('run_id', 'size'))
      .reset_index())
t['sdg_method'] = pd.Categorical(t.sdg_method, METHOD_ORDER, ordered=True)
t = t.sort_values(['dataset', 'sdg_method', 'eps_label'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, ds in zip(axes, DATASETS):
    d = t[t.dataset == ds]
    for m, sub in d.groupby('sdg_method', observed=True):
        ax.scatter(sub.tvd_2way, sub.lift, s=140, color=COLORS[m], label=nice(m),
                   edgecolor='k', linewidth=0.6, zorder=3)
        for _, r in sub.iterrows():
            ax.annotate('\u03b5=%s' % r.eps_label, (r.tvd_2way, r.lift),
                        fontsize=8, xytext=(5, 4), textcoords='offset points')
    ax.axhline(0, color='k', ls='--', lw=1.3)
    ax.text(ax.get_xlim()[1], 0.004, 'no better than guessing', ha='right', fontsize=9)
    ax.set_xlabel('2-way TVD (lower = better fidelity)')
    ax.set_ylabel('accuracy lift over majority baseline')
    ax.set_title('%s — causal GAN family' % nice(ds))
    ax.legend(fontsize=9, frameon=False)
fig.suptitle('Where the causal GANs land once collapsed runs are excluded',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
t.round(4).style.hide(axis='index')

**Read:**

- **The representation win moved datasets.** On Adult, DECAF+CTGAN is now a decisive fidelity
  win: 2-way TVD **0.179 against plain DECAF's 0.543**, with positive lift (+0.018) where
  plain DECAF is negative (−0.013). On COMPAS the ranking *reversed* versus the old batch —
  plain DECAF at 1000 epochs now reaches **0.077**, better than DECAF+CTGAN's 0.124. The
  CTGAN representation still wins COMPAS on utility (lift +0.060 vs +0.051), just not on
  fidelity. Representation and training length interact; neither can be tuned alone.
- **"Never collapses" is not the same as "useful", and DP-CTGAN shows the difference.**
  On Adult it survives every budget but lands at lift **−0.001 to −0.016** — it is
  reproducing the majority class and nothing more. Its apparently excellent fairness gap
  there (0.000–0.013) is the trivially-fair failure mode, not an achievement. The one private
  cell in the family that genuinely clears the baseline is **DP-CTGAN on COMPAS at
  ε=1000: lift +0.026, gap 0.159**.
- The same caution applies to DP-GAN's *survivors*: on Adult at ε=10 and ε=1000 the
  runs that did not collapse sit at lift +0.000 and gap 0.000. Excluding collapsed runs makes
  the table readable, but what is left is largely degenerate too.
- **The Adult DECAF+CTGAN row is now reliable** — this is the cell the epoch correction fixed
  (Section 4.3c), and it moved from 18 points below the trivial predictor to 2 above it.

## 9. Takeaways — what we can defend today

### The case that this is a paper

**1. The central hypothesis survived contact with 2,040 runs, including a full re-run of the
GAN family at corrected settings.** Removing a causal pathway at
generation time reduces the downstream demographic parity gap, and it costs approximately
nothing in accuracy or fidelity. The effect is measurable above a well-characterised noise
floor using matched pairs. This is the contribution the project was set up to make, and it
holds across two datasets, four generators, three role splits and five seeds.

**2. Privacy and fairness are near-independent here.** A 1000× change in ε moves the fairness
gap far less than the choice of mechanism does. The literature generally assumes a tension;
we can show that at this scale there mostly is not one. Negative results of this
specificity are publishable and useful.

**3. A concrete methods contribution: DP-CTGAN rescues private causal generation.** 41%
collapse → 0% collapse at identical privacy budgets, purely from the representation. This is
the most novel single finding and it stands on its own.

**4. A measurement contribution: the audit-error axis.** Whether a synthetic release reports
its *own* fairness honestly is measurable, is not routinely measured, and turns out to vary
a lot by generator. This is a natural second contribution and a good reviewer-proofing move.

**5. A methodological warning worth publishing.** Standard fidelity metrics (1-way and 2-way
TVD) completely failed to detect a generator producing *inverted* feature→outcome structure
(Section 4.3), and raw accuracy on an imbalanced dataset hid that several methods learned
nothing at all (Section 6). Both are easy mistakes that the field makes.

### What we cannot claim yet

- **We are not producing state-of-the-art synthetic data on Adult.** Most methods sit at or
  below the trivial baseline there. The fairness conclusions are comparative and survive this,
  but a utility claim does not.
- **The causal DAG is assumed, not discovered.** Every result is conditional on a hand-
  specified graph. A reviewer will press on this.
- **Two datasets, both small and fully categorical.** No continuous attributes, no
  high-cardinality real-world table.
- **A known encoding bug in Adult** (`education-num` is passed through with its raw UCI 1–16
  coding rather than being ordinal-coded like the other columns) is present in every Adult run
  in this batch. It is consistent across all methods so the comparisons are internally valid,
  but the absolute Adult numbers will move when it is fixed.

## 10. Next steps

Ordered by what most increases the chance this becomes a strong submission.

> ### ✅ Status update — items now in flight
>
> Since this notebook was first written, four of the items below have been actioned. They
> are left in the list with their reasoning intact, annotated with what changed:
>
> - **Item 1 (epoch sweep)** — ✅ **complete, and the re-run has landed.** All eight (GAN
>   method × dataset) combinations were swept, ranked on 2-way TVD over 3 seeds, and for the
>   private variants swept *jointly over epochs and ε* (114 fits). **Six of eight cells
>   changed.** The full grid was then re-run at the corrected settings as batch
>   `gan-retuned-2026-08-04` (960 runs; Adult 480/480 clean, COMPAS 479/480). Sections 4–8
>   are cut against it. Headline: Adult/DECAF+CTGAN moved from accuracy 0.569 to **0.766**
>   against a 0.747 baseline. See Section 4.3c.
> - **Item 3 (encoding bug)** — fixed for the two new datasets; deliberately **not** fixed on
>   Adult, because silently re-encoding would invalidate every published number rather than
>   improve it. Adult needs its own re-run under a fresh batch tag.
> - **Item 10 (more datasets)** — **done and running.** SNAKE (15 attributes) and NIST SBO
>   (25 attributes) are implemented with validated causal DAGs; see Section 1.1. SBO takes the
>   graph from Adult's 37 protected→outcome paths to **465**, which is the scaling test this
>   project most needs.
> - **Experiment tracking** — audited and repaired. The database was clean on the things that
>   would invalidate results (no duplicate configs, no orphan metrics, consistent metric
>   counts, zero failed runs in either batch), but three provenance defects were fixed: the
>   git fingerprint ignored uncommitted changes (so 227 GAN rows are tagged with a commit that
>   predates the code that made them), GAN rows recorded no epoch count at all, and 12
>   DECAF/COMPAS rows predate the output-head fix and sit at a 3× different fidelity from the
>   other 108 in their cell. See the package README.

### 10.1 Fix and re-run (highest priority, cheap)

1. ~~**Relaunch the causal GANs with longer training**~~ — ✅ **done** (Section 4.3c). What
   it leaves open: the sweep ranked on 3 seeds, and the 300-epoch result it first produced
   turned out to be a lucky draw that 5 seeds overturned. **The selection procedure itself is
   still under-powered.** Before the next tuning pass, decide how many seeds a configuration
   must win on. Also: COMPAS DP-GAN now collapses in **100% of runs at ε=1**, so that cell
   currently has no usable setting at all and should either be dropped from the grid or
   given a different critic.
2. **Fix the Adult `education-num` encoding**, then re-run Adult under a fresh batch tag.
3. **Add AIM to the method family.** It is implemented and registered but was left out of
   both batches; it is the strongest marginal-based baseline and its absence is conspicuous.
4. **Record the synthetic outcome marginal as a logged metric**, so the near-collapse cases
   (two classes, but 99/1) are visible without re-deriving them. Right now only full collapse
   is flagged. A follow-up probe on Adult/DECAF+CTGAN found the synthetic positive rate
   swinging between **0.09 and 0.47 across seeds against a true rate of 0.25** — a 5× spread
   that no logged metric would currently reveal.
5. **Log `spent_epsilon` for DP-CTGAN too.** The accountant trace is currently recorded only
   for DP-GAN, so the realised budget of the variant we most want to promote is not
   independently verifiable from the database.

### 10.2 Strengthen the science

6. **Report accuracy as lift over baseline everywhere**, and add a train-on-real reference
   row to every utility table. Section 6 shows how much this changes the reading.
7. **Add a proper privacy attack** — membership inference or attribute inference against the
   synthetic releases. ε is a worst-case bound; an empirical attack tells you what the actual
   exposure is, and the gap between them is interesting in its own right. Right now "privacy"
   is a parameter we set rather than a property we measure.
8. **Vary the causal graph deliberately.** Since every result is conditional on the assumed
   DAG, run the main grid under a perturbed graph (add/remove/reverse a few edges) and show
   the conclusions are robust. This directly answers the reviewer objection above.
9. **Sweep subgroup size.** The privacy–fairness independence result in Section 5 is the
   claim most likely to be attacked. Sub-sample a protected group down to 1%, 2%, 5% and
   show where the DP-hurts-minorities effect actually switches on.

### 10.3 Extend the scope

10. ~~**A third dataset**~~ — **done: SNAKE and SBO are added** (Section 1.1), taking the
   project from 2 datasets to 4 and from a maximum of 37 protected→outcome causal paths to
   **465**. That turns "does the mechanism work" into the sharper question **"does it still
   work when the graph is big enough that most paths are indirect?"** — which is exactly
   where CF should separate from DP, and where two small tables could never tell them apart.
   Still worth adding a dataset with genuinely *continuous* attributes (ACS / folktables);
   all four current tables are categorical.
11. **Compare against post-hoc fairness repair.** The implicit argument is that fixing bias at
    generation time beats fixing it at model-training time. Nobody has been made to prove that
    here — running a standard in-processing fair classifier on the *unmodified* synthetic data
    is the head-to-head the paper needs.
12. **Push DP-CTGAN harder.** It never collapsed, which means it has headroom: it is the only
    private causal generator that works, so tune it properly and see how close it can get to
    the marginal-based methods.
13. **Test whether the causal intervention transfers.** Every fairness number here is for one
    downstream classifier family. If the mechanism only helps MLPs, that is a much weaker
    claim than if it helps whatever a practitioner happens to train.

### 10.4 Questions worth deciding before the next batch

- Is the headline **"causal fairness is nearly free"** (needs the paired analysis front and
  centre) or **"private causal generation is possible"** (needs DP-CTGAN front and centre)?
  They are both supportable and they want different experiments next.
- Is the audit-error result a section, or its own paper?
- Do we need DECAF at all if DP-CTGAN dominates it on both fidelity and stability?

---

### Appendix: reproducing any number here

```
repo   : CausalFairnessInSDG
batches: overnight-2026-07-30      (1200 runs, marginal methods + DECAF)
         gan-retuned-2026-08-04    ( 960 runs, GAN family at swept epochs) [ACTIVE]
         gan-backbones-2026-07-30  ( 840 runs, superseded -- Section 4.3c only)
         newdata-2026-08-03        ( 216 runs, SNAKE + SBO, separate file)

all_runs.csv carries a `superseded` flag. The notebook analyses superseded == False;
only Section 4.3c reads the superseded rows, to show the before/after.
data   : all_runs.csv        -- one row per run, every metric, joinable on run_id
         real_baselines.csv  -- majority-class and train-on-real reference accuracies
```

Every row in `all_runs.csv` is a complete experiment: the axis columns identify the cell, and
everything else is a measurement. Nothing in this notebook re-runs a generator — it is all
aggregation over that table, so any figure can be regenerated or re-cut without the repo.